In [1]:
"""
================================================================================
程序4：非线性组完整流程（阶段0-8）- 修复版 v3.1 + 完整MAE/RMSE
================================================================================
专注于非线性组：SVR / KNN / MLP / AdaBoost

✅ 核心优化（v3.1）：
1. 移除样本扰动策略
2. 添加L2正则化（MLP alpha=0.01）
3. 简化MLP结构
4. 优化AdaBoost参数
5. 区分确定性/随机性模型
6. 新增训练集和测试集MAE/RMSE完整计算、打印和保存 ← 本次新增

版本：v3.1 + 完整MAE/RMSE
日期：2024
================================================================================
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.base import clone

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mutual_info_score
from scipy.stats import pearsonr, spearmanr, rankdata
import os
import warnings
import pickle
import time
import itertools
from collections import Counter, defaultdict
import multiprocessing
from joblib import Parallel, delayed

try:
    from dcor import distance_correlation
    DCOR_AVAILABLE = True
except ImportError:
    DCOR_AVAILABLE = False
    print("Warning: dcor not available, distance correlation will not be calculated")

warnings.filterwarnings('ignore')

n_cores = multiprocessing.cpu_count()
print(f"检测到 {n_cores} 个CPU核心")

# ==================== 配置参数 ====================
CONFIG = {
    'data_file': "D:\\博一下\\第一章主动学习\\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx",
    'output_dir': "D:\\博一下\\第一章主动学习\\非线性组_结果_优化版_v3.1-4.22",
    'target_col': 'KQ',
    'exclude_cols': ['ID', 'Sample_Name', 'Notes'],

    'stage1_target': 180,
    'stage2_target': 60,
    'stage4_nonlinear_target': 18,

    'min_r2_threshold': 0.50,
    'max_r2_threshold': 0.95,
    'max_generalization_gap': 0.25,

    'deterministic_models': ['SVR', 'KNN'],
    'stochastic_models': ['MLP', 'AdaBoost'],

    'n_repeat_seeds': 5,
    'base_random_state': 2023,
    'stability_penalty': 0.10,

    'random_state': 2023,
    'test_size': 0.2,
    'n_jobs': 3,
    'stage6_n_jobs': 6,

    'randomization_strategy': None
}

CONFIG['cv_strategy'] = {
    'stage5': {
        'method': 'kfold',
        'n_splits': 5,
        'shuffle': True,
        'random_state': 2023
    },
    'stage6': {
        'method': 'repeated_kfold',
        'n_splits': 5,
        'n_repeats': 3,
        'random_state': 2023
    },
    'stage8': {
        'method': 'repeated_kfold',
        'n_splits': 5,
        'n_repeats': 5,
        'random_state': 2023
    }
}

CONFIG['convergence_check'] = {
    'mlp_max_iter': 1000,
    'mlp_early_stopping': True,
    'mlp_validation_fraction': 0.2,
    'mlp_tol': 1e-4,
    'mlp_n_iter_no_change': 20,
    'svr_max_iter': -1,
    'svr_tol': 1e-3
}

CONFIG['feature_tiers'] = {
    'S': {
        'features': ['Nb', 'Si', 'VEC', 'δ'],
        'weight': 1.5,
        'description': 'S级(核心特征)'
    },
    'A': {
        'features': [
            'x', 'Δx', 'ΔHmix', 'ΔSmix', 'ΔG', 'Ω', 'Λ', 'Tm', 'ΔTm',
            'ρ', 'r', 'am', 'Δa', 'D.χ', 'D.r', 'Ec',
            'Pugh_modulus_ratio', 'enthalpy_entropy_competition',
            'phase_stability_index', 'Nb_Si_ratio', 'eutectic_distance',
            'VEC_Omega_coupling'
        ],
        'weight': 1.2,
        'description': 'A级(重要特征)'
    },
    'B': {
        'weight': 1.0,
        'description': 'B级(标准特征)'
    }
}

CONFIG['stage1'] = {
    'unique_ratio_threshold': 0.05,
    'high_collinearity_threshold': 0.98,
}

CONFIG['stage2'] = {
    'weights': {
        'pearson': 0.45,
        'mi': 0.35,
        'discriminative': 0.20
    },
    'weights_with_dcor': {
        'pearson': 0.35,
        'mi': 0.30,
        'dcor': 0.20,
        'discriminative': 0.15
    },
    'tier_multipliers': {
        'S': 1.5,
        'A': 1.2,
        'B': 1.0
    },
    'mi_discretization': {
        'method': 'quantile',
        'n_bins': 10,
        'strategy': 'discrete'
    }
}

CONFIG['stage5'] = {
    'k_range': [4, 5, 6, 7],
    'top_n_per_k': 5,
    'n_jobs_parallel': 5
}

# 创建输出目录
os.makedirs(CONFIG['output_dir'], exist_ok=True)
predictions_dir = os.path.join(CONFIG['output_dir'], 'detailed_predictions')
os.makedirs(predictions_dir, exist_ok=True)
top100_dir = os.path.join(CONFIG['output_dir'], 'top100_models')
os.makedirs(top100_dir, exist_ok=True)
top100_predictions_dir = os.path.join(top100_dir, 'predictions')
os.makedirs(top100_predictions_dir, exist_ok=True)
top100_models_dir = os.path.join(top100_dir, 'model_objects')
os.makedirs(top100_models_dir, exist_ok=True)
top30_dir = os.path.join(CONFIG['output_dir'], 'top30_models')
os.makedirs(top30_dir, exist_ok=True)
top30_predictions_dir = os.path.join(top30_dir, 'predictions')
os.makedirs(top30_predictions_dir, exist_ok=True)
top30_models_dir = os.path.join(top30_dir, 'model_objects')
os.makedirs(top30_models_dir, exist_ok=True)
stage8_dir = os.path.join(CONFIG['output_dir'], 'stage8_ranking_consistency')
os.makedirs(stage8_dir, exist_ok=True)


# ==================== 数据清洗函数 ====================
def check_and_remove_invalid_values(X, y, data_name="数据"):
    """检查并移除包含 NaN 或 Inf/-Inf 的样本"""
    print(f"\n  检查{data_name}中的无效值...")

    n_samples_before = X.shape[0]

    nan_mask_x = ~np.isnan(X).any(axis=1)
    nan_mask_y = ~np.isnan(y)
    nan_removed = n_samples_before - np.sum(nan_mask_x & nan_mask_y)

    if nan_removed > 0:
        print(f"    ⚠️  发现 {nan_removed} 个含NaN的样本")

    inf_mask_x = ~np.isinf(X).any(axis=1)
    inf_mask_y = ~np.isinf(y)
    inf_removed = n_samples_before - np.sum(inf_mask_x & inf_mask_y)

    if inf_removed > 0:
        print(f"    ⚠️  发现 {inf_removed} 个含Inf/-Inf的样本")

    valid_mask = nan_mask_x & nan_mask_y & inf_mask_x & inf_mask_y

    X_clean = X[valid_mask]
    y_clean = y[valid_mask]
    valid_indices = np.where(valid_mask)[0]

    n_samples_after = X_clean.shape[0]
    n_removed_total = n_samples_before - n_samples_after

    if n_removed_total > 0:
        print(f"    ✅ 移除 {n_removed_total} 个无效样本 "
              f"({n_samples_before} → {n_samples_after})")
    else:
        print(f"    ✅ 数据clean，无无效值")

    return X_clean, y_clean, valid_indices


# ==================== 辅助函数 ====================
def create_stratify_bins(y, n_bins=5):
    """创建分层标签"""
    print(f"\n  创建分层标签({n_bins}个区间):")
    percentiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(y, percentiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    stratify_labels = np.digitize(y, bin_edges) - 1

    for i in range(n_bins):
        mask = stratify_labels == i
        count = np.sum(mask)
        if count > 0:
            y_in_bin = y[mask]
            print(f"    区间 {i+1}: KQ范围 [{y_in_bin.min():.2f}, {y_in_bin.max():.2f}], "
                  f"样本数={count}, 均值={y_in_bin.mean():.2f}")

    return stratify_labels

def assign_feature_tiers(features_name):
    """为所有特征分配等级标签"""
    print("\n==================== 特征分级标记 ====================")

    feature_tier_map = {}
    s_tier_features = CONFIG['feature_tiers']['S']['features']
    a_tier_features = CONFIG['feature_tiers']['A']['features']

    s_count = 0
    a_count = 0
    b_count = 0

    for feat in features_name:
        if feat in s_tier_features:
            feature_tier_map[feat] = 'S'
            s_count += 1
        elif feat in a_tier_features:
            feature_tier_map[feat] = 'A'
            a_count += 1
        else:
            feature_tier_map[feat] = 'B'
            b_count += 1

    print(f"\n特征分级统计:")
    print(f"  S级: {s_count}个 - {CONFIG['feature_tiers']['S']['description']}")
    print(f"  A级: {a_count}个 - {CONFIG['feature_tiers']['A']['description']}")
    print(f"  B级: {b_count}个 - {CONFIG['feature_tiers']['B']['description']}")

    return feature_tier_map

def calculate_comprehensive_metrics(y_true, y_pred):
    """计算全面的回归指标"""
    try:
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)

        y_true_nonzero = np.where(np.abs(y_true) < 1e-6, 1e-6, y_true)
        mape = np.mean(np.abs((y_true - y_pred) / y_true_nonzero)) * 100

        y_true_pos = np.where(y_true <= 0, 1e-6, y_true)
        y_pred_pos = np.where(y_pred <= 0, 1e-6, y_pred)
        rmsle = np.sqrt(mean_squared_log_error(y_true_pos, y_pred_pos))

        if np.isnan(r2) or np.isinf(r2):
            r2 = -float('inf')
        if np.isnan(rmse) or np.isinf(rmse):
            rmse = float('inf')
        if np.isnan(mae) or np.isinf(mae):
            mae = float('inf')
        if np.isnan(mape) or np.isinf(mape):
            mape = float('inf')
        if np.isnan(rmsle) or np.isinf(rmsle):
            rmsle = float('inf')

        return {
            'r2': r2, 'rmse': rmse, 'mae': mae,
            'mape': mape, 'rmsle': rmsle
        }
    except:
        return {
            'r2': -float('inf'), 'rmse': float('inf'), 'mae': float('inf'),
            'mape': float('inf'), 'rmsle': float('inf')
        }

def save_detailed_predictions(model_key, y_train, pred_train, y_test, pred_test,
                            train_indices, test_indices, timestamp):
    """保存模型的详细预测结果"""
    train_results = pd.DataFrame({
        'Dataset': ['Training'] * len(y_train),
        'Dataset_Code': [1] * len(y_train),
        'Sample_Index': train_indices + 1,
        'Experimental_KQ': y_train,
        'Predicted_KQ': pred_train,
        'Error': pred_train - y_train,
        'Absolute_Error': np.abs(pred_train - y_train),
        'Relative_Error_Percent': np.abs(pred_train - y_train) / np.abs(y_train) * 100
    })

    test_results = pd.DataFrame({
        'Dataset': ['Test'] * len(y_test),
        'Dataset_Code': [2] * len(y_test),
        'Sample_Index': test_indices + 1,
        'Experimental_KQ': y_test,
        'Predicted_KQ': pred_test,
        'Error': pred_test - y_test,
        'Absolute_Error': np.abs(pred_test - y_test),
        'Relative_Error_Percent': np.abs(pred_test - y_test) / np.abs(y_test) * 100
    })

    all_results = pd.concat([train_results, test_results], ignore_index=True)
    filename = f"{model_key}_predictions_{timestamp}.csv"
    filepath = os.path.join(predictions_dir, filename)
    all_results.to_csv(filepath, index=False, encoding='utf-8')
    return filepath


# ==================== 统一的交叉验证函数 ====================
def perform_cross_validation(model, X, y, cv_config, need_scaling=True):
    """使用普通KFold，无数据泄露"""
    method = cv_config.get('method', 'kfold')

    if method == 'repeated_kfold':
        cv_splitter = RepeatedKFold(
            n_splits=cv_config['n_splits'],
            n_repeats=cv_config.get('n_repeats', 3),
            random_state=cv_config['random_state']
        )
    else:
        cv_splitter = KFold(
            n_splits=cv_config['n_splits'],
            shuffle=cv_config.get('shuffle', True),
            random_state=cv_config.get('random_state', 2023)
        )

    splits = cv_splitter.split(X)

    cv_scores = []
    fold_predictions = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_train_fold = X[train_idx]
        X_val_fold = X[val_idx]
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]

        if need_scaling:
            scaler = StandardScaler()
            X_train_fold = scaler.fit_transform(X_train_fold)
            X_val_fold = scaler.transform(X_val_fold)

        try:
            model.fit(X_train_fold, y_train_fold)
            val_pred = model.predict(X_val_fold)
            val_score = r2_score(y_val_fold, val_pred)
            cv_scores.append(val_score)

            fold_predictions.append({
                'fold': fold_idx,
                'val_indices': val_idx,
                'y_true': y_val_fold,
                'y_pred': val_pred,
                'score': val_score
            })
        except Exception as e:
            continue

    if len(cv_scores) == 0:
        return None

    return {
        'mean_score': np.mean(cv_scores),
        'std_score': np.std(cv_scores),
        'all_scores': cv_scores,
        'n_splits': len(cv_scores),
        'fold_predictions': fold_predictions,
        'cv_method': method
    }


# ==================== 阶段1：快速粗筛 ====================
def stage1_quick_filter(X_train, y_train, features_name, feature_tier_map):
    """阶段1：快速粗筛（纯规则）"""
    print("\n" + "="*80)
    print("【阶段1】快速粗筛 - 删除明显垃圾特征（纯规则）")
    print("="*80)
    print(f"输入特征数: {X_train.shape[1]}")

    start_time = time.time()

    keep_mask = np.ones(X_train.shape[1], dtype=bool)
    removal_log = []

    print(f"\n步骤1.1：删除常数/近常数特征")
    print(f"  条件：唯一值比例 < {CONFIG['stage1']['unique_ratio_threshold']}")

    removed_11_features = []
    for i, feat in enumerate(features_name):
        unique_ratio = len(np.unique(X_train[:, i])) / len(X_train[:, i])
        tier = feature_tier_map[feat]

        if tier == 'S':
            continue

        if unique_ratio < CONFIG['stage1']['unique_ratio_threshold']:
            keep_mask[i] = False
            removal_log.append({
                'stage': '1.1',
                'feature': feat,
                'tier': tier,
                'reason': f'近常数特征(唯一值比例={unique_ratio:.4f})'
            })
            removed_11_features.append((feat, tier, unique_ratio))

    removed_count_11 = len(removed_11_features)
    print(f"  删除: {removed_count_11}个")

    print(f"\n步骤1.2：删除全零特征")

    removed_12_features = []
    for i, feat in enumerate(features_name):
        if not keep_mask[i]:
            continue

        if np.all(X_train[:, i] == 0):
            keep_mask[i] = False
            tier = feature_tier_map[feat]
            removal_log.append({
                'stage': '1.2',
                'feature': feat,
                'tier': tier,
                'reason': '全零特征'
            })
            removed_12_features.append((feat, tier))

    removed_count_12 = (~keep_mask).sum()
    print(f"  删除: {removed_count_12 - removed_count_11}个")

    print(f"\n步骤1.3：删除极度共线特征")
    print(f"  阈值：|相关系数| > {CONFIG['stage1']['high_collinearity_threshold']}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    corr_matrix = np.corrcoef(X_scaled, rowvar=False)

    threshold = CONFIG['stage1']['high_collinearity_threshold']
    tier_order = {'S': 3, 'A': 2, 'B': 1}

    removed_13_features = []
    for i in range(len(features_name)):
        if not keep_mask[i]:
            continue
        for j in range(i+1, len(features_name)):
            if not keep_mask[j]:
                continue

            if abs(corr_matrix[i, j]) > threshold:
                feat_i = features_name[i]
                feat_j = features_name[j]
                tier_i = feature_tier_map[feat_i]
                tier_j = feature_tier_map[feat_j]

                if tier_order[tier_i] > tier_order[tier_j]:
                    keep_mask[j] = False
                    reason = f'与{feat_i}极度共线(r={corr_matrix[i,j]:.3f}), {tier_j}级<{tier_i}级'
                    removal_log.append({
                        'stage': '1.3', 'feature': feat_j,
                        'tier': tier_j, 'reason': reason
                    })
                    removed_13_features.append((feat_j, tier_j, feat_i, corr_matrix[i,j]))

                elif tier_order[tier_i] < tier_order[tier_j]:
                    keep_mask[i] = False
                    reason = f'与{feat_j}极度共线(r={corr_matrix[i,j]:.3f}), {tier_i}级<{tier_j}级'
                    removal_log.append({
                        'stage': '1.3', 'feature': feat_i,
                        'tier': tier_i, 'reason': reason
                    })
                    removed_13_features.append((feat_i, tier_i, feat_j, corr_matrix[i,j]))
                    break

                else:
                    r_i = abs(np.corrcoef(X_scaled[:, i], y_train)[0, 1])
                    r_j = abs(np.corrcoef(X_scaled[:, j], y_train)[0, 1])

                    if r_i > r_j:
                        keep_mask[j] = False
                        reason = f'与{feat_i}极度共线(r={corr_matrix[i,j]:.3f}), 同级但与目标相关性低'
                        removal_log.append({
                            'stage': '1.3', 'feature': feat_j,
                            'tier': tier_j, 'reason': reason
                        })
                        removed_13_features.append((feat_j, tier_j, feat_i, corr_matrix[i,j]))
                    else:
                        keep_mask[i] = False
                        reason = f'与{feat_j}极度共线(r={corr_matrix[i,j]:.3f}), 同级但与目标相关性低'
                        removal_log.append({
                            'stage': '1.3', 'feature': feat_i,
                            'tier': tier_i, 'reason': reason
                        })
                        removed_13_features.append((feat_i, tier_i, feat_j, corr_matrix[i,j]))
                        break

    removed_count_13 = (~keep_mask).sum()
    print(f"  删除: {removed_count_13 - removed_count_12}个")

    keep_indices = np.where(keep_mask)[0].tolist()
    selected_features = [features_name[i] for i in keep_indices]

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段1完成统计:")
    print("="*80)
    print(f"输入特征数: {len(features_name)}")
    print(f"保留特征数: {len(selected_features)}")
    print(f"删除特征数: {len(removal_log)}")
    print(f"\n⏱️  耗时: {elapsed_time:.1f}秒")

    if removal_log:
        report_df = pd.DataFrame(removal_log)
        report_path = os.path.join(CONFIG['output_dir'], 'stage1_quick_filter_report.csv')
        report_df.to_csv(report_path, index=False, encoding='utf-8')
        print(f"\n阶段1删除详细报告已保存: {report_path}")

    return keep_indices, selected_features


# ==================== 阶段2：NMI计算 ====================
def calculate_normalized_mutual_information(X_scaled, y_train, n_bins=10, method='quantile'):
    """使用标准化互信息（NMI）计算特征间冗余度"""
    n_features = X_scaled.shape[1]

    print(f"    正在离散化特征 (方法={method}, bins={n_bins})...")
    X_discrete = np.zeros_like(X_scaled, dtype=int)

    for i in range(n_features):
        feature_data = X_scaled[:, i]

        if method == 'quantile':
            try:
                X_discrete[:, i] = pd.qcut(feature_data, q=n_bins, labels=False, duplicates='drop')
            except:
                X_discrete[:, i] = pd.cut(feature_data, bins=n_bins, labels=False)
        else:
            X_discrete[:, i] = pd.cut(feature_data, bins=n_bins, labels=False)

        if np.isnan(X_discrete[:, i]).any():
            mode_val = pd.Series(X_discrete[:, i]).mode()[0]
            X_discrete[:, i] = np.nan_to_num(X_discrete[:, i], nan=mode_val)

    print(f"    离散化完成，开始计算 {n_features}×{n_features} NMI矩阵...")

    entropies = np.zeros(n_features)
    for i in range(n_features):
        entropies[i] = mutual_info_score(X_discrete[:, i], X_discrete[:, i])

    nmi_matrix = np.zeros((n_features, n_features))

    for i in range(n_features):
        nmi_matrix[i, i] = 1.0

        for j in range(i+1, n_features):
            mi_ij = mutual_info_score(X_discrete[:, i], X_discrete[:, j])
            H_i = entropies[i]
            H_j = entropies[j]

            if (H_i + H_j) > 0:
                nmi = 2.0 * mi_ij / (H_i + H_j)
            else:
                nmi = 0.0

            nmi_matrix[i, j] = nmi
            nmi_matrix[j, i] = nmi

        if (i+1) % 20 == 0:
            print(f"      进度: {i+1}/{n_features}")

    print(f"    ✅ NMI矩阵计算完成（使用标准化互信息）")

    return nmi_matrix


# ==================== 阶段2：综合筛选 ====================
def stage2_integrated_selection(X_train, y_train, features_name, feature_tier_map):
    """阶段2：综合筛选（使用NMI计算）"""
    print("\n" + "="*80)
    print("【阶段2】综合筛选 - 评分+去冗余一步完成（模型不可知）")
    print("="*80)
    print(f"输入特征数: {X_train.shape[1]}")

    start_time = time.time()
    target_n = CONFIG['stage2_target']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    print("\n" + "-"*80)
    print("步骤2.1：计算Relevance（模型不可知的多维度评分）")
    print("-"*80)

    n_features = X_train.shape[1]

    print("\n  指标1：计算Pearson相关（45%权重）...")
    pearson_scores = []
    for i in range(n_features):
        r, _ = pearsonr(X_scaled[:, i], y_train)
        pearson_scores.append(abs(r))
    pearson_scores = np.array(pearson_scores)

    print("  指标2：计算互信息MI（35%权重）...")
    mi_scores = mutual_info_regression(X_scaled, y_train, random_state=CONFIG['random_state'])

    print("  指标3：计算目标条件方差（20%权重）...")
    n_bins = 5
    y_percentiles = np.percentile(y_train, np.linspace(0, 100, n_bins + 1))

    discriminative_scores = []
    for i in range(n_features):
        group_vars = []
        for j in range(n_bins):
            mask = (y_train >= y_percentiles[j]) & (y_train < y_percentiles[j+1])
            if np.sum(mask) > 1:
                group_vars.append(np.var(X_scaled[mask, i]))

        if len(group_vars) > 0:
            discriminative_scores.append(np.std(group_vars))
        else:
            discriminative_scores.append(0)

    discriminative_scores = np.array(discriminative_scores)

    dcor_scores = None
    if DCOR_AVAILABLE:
        print("  可选指标：计算距离相关dCor（20%权重）...")
        dcor_scores = []
        for i in range(n_features):
            dcor = distance_correlation(X_scaled[:, i], y_train)
            dcor_scores.append(dcor)
        dcor_scores = np.array(dcor_scores)

    print("\n  Rank归一化...")
    pearson_ranks = rankdata(pearson_scores) / n_features
    mi_ranks = rankdata(mi_scores) / n_features
    discriminative_ranks = rankdata(discriminative_scores) / n_features

    if dcor_scores is not None:
        dcor_ranks = rankdata(dcor_scores) / n_features

    if dcor_scores is not None:
        weights = CONFIG['stage2']['weights_with_dcor']
        base_scores = (weights['pearson'] * pearson_ranks +
                      weights['mi'] * mi_ranks +
                      weights['dcor'] * dcor_ranks +
                      weights['discriminative'] * discriminative_ranks)
    else:
        weights = CONFIG['stage2']['weights']
        base_scores = (weights['pearson'] * pearson_ranks +
                      weights['mi'] * mi_ranks +
                      weights['discriminative'] * discriminative_ranks)

    print("\n  应用等级加权...")
    tier_multipliers = CONFIG['stage2']['tier_multipliers']

    relevance = base_scores.copy()
    for i, feat in enumerate(features_name):
        tier = feature_tier_map[feat]
        multiplier = tier_multipliers[tier]
        relevance[i] = base_scores[i] * multiplier

    print("\n" + "-"*80)
    print("步骤2.2：mRMR迭代选择（用NMI度量冗余）")
    print("-"*80)
    print(f"  目标选择: {target_n}个特征")

    print("\n  ✅ 使用NMI（标准化互信息）计算特征间冗余度...")
    mi_config = CONFIG['stage2']['mi_discretization']

    nmi_matrix = calculate_normalized_mutual_information(
        X_scaled,
        y_train,
        n_bins=mi_config['n_bins'],
        method=mi_config['method']
    )

    print("\n  开始mRMR迭代选择...")
    selected_indices = []
    remaining_indices = list(range(n_features))

    first_idx = np.argmax(relevance)
    selected_indices.append(first_idx)
    remaining_indices.remove(first_idx)

    print(f"    第1/{target_n}个: {features_name[first_idx]} (relevance={relevance[first_idx]:.4f})")

    for step in range(1, target_n):
        best_score = -np.inf
        best_idx = None

        for idx in remaining_indices:
            redundancy = np.mean([nmi_matrix[idx, s] for s in selected_indices])
            mrmr_score = relevance[idx] - redundancy

            if mrmr_score > best_score:
                best_score = mrmr_score
                best_idx = idx

        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)

        if (step + 1) % 10 == 0 or (step + 1) == target_n:
            redundancy_val = np.mean([nmi_matrix[best_idx, s] for s in selected_indices[:-1]])
            print(f"    第{step+1}/{target_n}个: {features_name[best_idx]} "
                  f"(relevance={relevance[best_idx]:.4f}, redundancy={redundancy_val:.4f})")

    selected_features = [features_name[i] for i in selected_indices]

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段2完成统计:")
    print("="*80)
    print(f"输入特征数: {len(features_name)}")
    print(f"输出特征数: {len(selected_features)}")
    print(f"\n⏱️  耗时: {elapsed_time:.1f}秒")

    return selected_indices, selected_features


# ==================== 阶段4：非线性组特征选择 ====================
def stage4_nonlinear_model_selection(X_train, y_train, features_60, feature_tier_map):
    """阶段4：非线性组特征选择（专版）"""
    print("\n" + "="*80)
    print("【阶段4】非线性组特征选择（专版）")
    print("="*80)
    print(f"输入特征数: {X_train.shape[1]}")
    print(f"目标选择: {CONFIG['stage4_nonlinear_target']}个特征")

    start_time = time.time()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    print("\n  方法1：计算距离相关dCor（非线性关系检测）（40%权重）...")

    if DCOR_AVAILABLE:
        dcor_scores = []
        for i in range(X_train.shape[1]):
            dcor = distance_correlation(X_scaled[:, i], y_train)
            dcor_scores.append(dcor)
        dcor_scores = np.array(dcor_scores)
        print(f"    ✓ 距离相关计算完成")
    else:
        print("    ⚠️  dcor库不可用，使用Pearson相关作为替代")
        dcor_scores = []
        for i in range(X_train.shape[1]):
            r, _ = pearsonr(X_scaled[:, i], y_train)
            dcor_scores.append(abs(r))
        dcor_scores = np.array(dcor_scores)

    print("\n  方法2：计算互信息MI（非线性依赖性）（35%权重）...")
    mi_scores = mutual_info_regression(X_scaled, y_train, random_state=CONFIG['random_state'])
    print(f"    ✓ 互信息计算完成")

    print("\n  方法3：计算Spearman秩相关（单调非线性关系）（25%权重）...")
    spearman_scores = []
    for i in range(X_train.shape[1]):
        rho, _ = spearmanr(X_scaled[:, i], y_train)
        spearman_scores.append(abs(rho))
    spearman_scores = np.array(spearman_scores)
    print(f"    ✓ Spearman相关计算完成")

    print("\n  归一化各方法的得分...")
    dcor_norm = (dcor_scores - dcor_scores.min()) / (dcor_scores.max() - dcor_scores.min() + 1e-10)
    mi_norm = (mi_scores - mi_scores.min()) / (mi_scores.max() - mi_scores.min() + 1e-10)
    spearman_norm = (spearman_scores - spearman_scores.min()) / (spearman_scores.max() - spearman_scores.min() + 1e-10)

    base_score_nonlinear = 0.4 * dcor_norm + 0.35 * mi_norm + 0.25 * spearman_norm

    print("\n  应用特征等级权重...")
    weighted_nonlinear = []
    for i, feat in enumerate(features_60):
        tier = feature_tier_map[feat]
        weight = CONFIG['feature_tiers'][tier]['weight']
        weighted_score = base_score_nonlinear[i] * weight
        weighted_nonlinear.append({
            'index': i,
            'feature': feat,
            'tier': tier,
            'base_score': base_score_nonlinear[i],
            'weighted_score': weighted_score,
            'dcor': dcor_scores[i],
            'mi': mi_scores[i],
            'spearman': spearman_scores[i]
        })

    weighted_nonlinear.sort(key=lambda x: x['weighted_score'], reverse=True)

    nonlinear_group_top = weighted_nonlinear[:CONFIG['stage4_nonlinear_target']]
    nonlinear_group_indices = [item['index'] for item in nonlinear_group_top]
    nonlinear_group_features = [item['feature'] for item in nonlinear_group_top]

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段4完成统计（非线性组）:")
    print("="*80)
    print(f"输入特征数: {len(features_60)}")
    print(f"输出特征数: {len(nonlinear_group_features)}")
    print(f"\n⏱️  耗时: {elapsed_time:.1f}秒")

    report_df = pd.DataFrame(weighted_nonlinear)
    report_path = os.path.join(CONFIG['output_dir'], 'stage4_nonlinear_group_feature_selection.csv')
    report_df.to_csv(report_path, index=False, encoding='utf-8')
    print(f"\n非线性组特征选择详细报告已保存: {report_path}")

    return {
        'indices': nonlinear_group_indices,
        'features': nonlinear_group_features,
        'detailed_scores': nonlinear_group_top
    }


# ==================== 阶段5：非线性组穷举搜索 ====================
def evaluate_combination_nonlinear_group(feature_indices, X_subset, y_train, random_state):
    """阶段5专用：用简单非线性模型评估非线性组特征组合"""
    np.random.seed(random_state)

    X_comb = X_subset[:, list(feature_indices)]

    models = [
        ('SVR', SVR(C=1.0, epsilon=0.1, kernel='rbf', gamma='scale')),
        ('KNN', KNeighborsRegressor(n_neighbors=5, weights='distance', metric='minkowski'))
    ]

    need_scaling = True
    cv_config = CONFIG['cv_strategy']['stage5']
    all_model_scores = []

    for model_name, model in models:
        try:
            cv_result = perform_cross_validation(
                model, X_comb, y_train, cv_config, need_scaling=need_scaling
            )
            if cv_result is not None:
                all_model_scores.append(cv_result['mean_score'])
        except Exception as e:
            continue

    if len(all_model_scores) > 0:
        ensemble_score = np.mean(all_model_scores)
    else:
        ensemble_score = -np.inf

    return {
        'feature_indices': list(feature_indices),
        'cv_r2': ensemble_score
    }

def stage5_nonlinear_group_exhaustive_search(X_train_subset, y_train, group_features):
    """阶段5：非线性组K=4-7穷举搜索（专版）"""
    print("\n" + "="*80)
    print("【阶段5】非线性组K=4-7穷举搜索（专版）")
    print("="*80)
    print(f"特征池: {len(group_features)}个特征")
    print(f"K值范围: {CONFIG['stage5']['k_range']}")
    print(f"并行进程数: {CONFIG['stage5']['n_jobs_parallel']}")

    start_time = time.time()

    results_all = []
    n_jobs_parallel = CONFIG['stage5']['n_jobs_parallel']

    for k in CONFIG['stage5']['k_range']:
        from math import comb
        n_combinations = comb(len(group_features), k)
        print(f"\n  K={k}: 共{n_combinations}个组合", end="", flush=True)

        k_start_time = time.time()

        all_combinations = list(itertools.combinations(range(len(group_features)), k))

        k_results_raw = Parallel(n_jobs=n_jobs_parallel, backend='loky', verbose=0)(
            delayed(evaluate_combination_nonlinear_group)(
                comb, X_train_subset, y_train, CONFIG['random_state']
            ) for comb in all_combinations
        )

        k_results = []
        for result in k_results_raw:
            selected_feats = [group_features[i] for i in result['feature_indices']]
            k_results.append({
                'k': k,
                'feature_indices': result['feature_indices'],
                'features': selected_feats,
                'cv_r2': result['cv_r2']
            })

        k_results.sort(key=lambda x: x['cv_r2'], reverse=True)

        top_n = min(CONFIG['stage5']['top_n_per_k'], len(k_results))
        k_top_results = k_results[:top_n]

        results_all.extend(k_top_results)

        k_elapsed = time.time() - k_start_time
        print(f" → Top{top_n}组合, 最佳CV R²={k_top_results[0]['cv_r2']:.4f} "
              f"(耗时{k_elapsed:.1f}秒)", flush=True)

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段5完成统计（非线性组）:")
    print("="*80)
    print(f"总评估组合数: 共{len(results_all)}个最优组合")
    print(f"⏱️  总耗时: {elapsed_time/60:.1f}分钟")

    results_df = pd.DataFrame([
        {
            'K值': r['k'],
            'CV_R2': r['cv_r2'],
            '特征数量': len(r['features']),
            '特征列表': ', '.join(r['features'])
        } for r in results_all
    ])
    results_path = os.path.join(CONFIG['output_dir'], 'stage5_nonlinear_group_combinations.csv')
    results_df.to_csv(results_path, index=False, encoding='utf-8')
    print(f"\n非线性组穷举结果已保存: {results_path}")

    return results_all


# ==================== 阶段6：非线性组最终训练 ====================
# ✅ 新增MAE/RMSE：新增训练集和测试集MAE/RMSE收集
def evaluate_single_nonlinear_model_stage6(args):
    """阶段6：评估单个非线性模型（修复版 - 区分确定性/随机性模型）"""
    (X_train_comb, X_test_comb, y_train, y_test,
     train_indices, test_indices, model_name, base_model,
     k, comb_idx, feature_names,
     cv_config, timestamp) = args

    model_key = f"Nonlinear_{model_name}_K{k}_Comb{comb_idx+1}"

    is_deterministic = model_name in CONFIG['deterministic_models']

    if is_deterministic:
        n_evaluations = 1
        repeat_seeds = [CONFIG['base_random_state']]
    else:
        n_evaluations = CONFIG['n_repeat_seeds']
        repeat_seeds = [CONFIG['base_random_state'] + i for i in range(n_evaluations)]

    cv_r2_list = []
    train_r2_list = []
    train_mae_list = []    # ✅ 新增MAE/RMSE
    train_rmse_list = []   # ✅ 新增MAE/RMSE
    test_r2_list = []
    test_mae_list = []
    test_rmse_list = []    # ✅ 新增MAE/RMSE
    gap_list = []
    convergence_warnings = []

    need_scaling = True

    for seed_idx, seed in enumerate(repeat_seeds):
        model = clone(base_model)

        if not is_deterministic:
            if hasattr(model, 'random_state'):
                try:
                    model.set_params(random_state=seed)
                except ValueError:
                    pass

        X_train_current = X_train_comb
        y_train_current = y_train

        cv_result = perform_cross_validation(
            model, X_train_current, y_train_current, cv_config,
            need_scaling=need_scaling
        )

        if cv_result is None:
            continue

        cv_r2_list.append(cv_result['mean_score'])

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_current)
        X_test_scaled = scaler.transform(X_test_comb)

        try:
            model.fit(X_train_scaled, y_train_current)

            if model_name == 'MLP' and hasattr(model, 'n_iter_'):
                max_iter = CONFIG['convergence_check']['mlp_max_iter']
                if model.n_iter_ >= max_iter - 1:
                    convergence_warnings.append(f"seed{seed}可能未收敛")

            pred_train = model.predict(X_train_scaled)
            pred_test = model.predict(X_test_scaled)

            train_metrics = calculate_comprehensive_metrics(y_train, pred_train)
            test_metrics = calculate_comprehensive_metrics(y_test, pred_test)

            train_r2_list.append(train_metrics['r2'])
            train_mae_list.append(train_metrics['mae'])    # ✅ 新增MAE/RMSE
            train_rmse_list.append(train_metrics['rmse'])  # ✅ 新增MAE/RMSE
            test_r2_list.append(test_metrics['r2'])
            test_mae_list.append(test_metrics['mae'])
            test_rmse_list.append(test_metrics['rmse'])    # ✅ 新增MAE/RMSE
            gap_list.append(train_metrics['r2'] - test_metrics['r2'])

            if seed_idx == 0:
                first_model = model
                first_scaler = scaler
                first_pred_train = pred_train
                first_pred_test = pred_test

        except Exception as e:
            continue

    min_required = 1 if is_deterministic else (n_evaluations // 2)
    if len(cv_r2_list) < min_required:
        return None

    mean_cv_r2 = np.mean(cv_r2_list)
    std_cv_r2 = np.std(cv_r2_list) if len(cv_r2_list) > 1 else 0.0
    mean_train_r2 = np.mean(train_r2_list)
    std_train_r2 = np.std(train_r2_list) if len(train_r2_list) > 1 else 0.0
    mean_train_mae = np.mean(train_mae_list)    # ✅ 新增MAE/RMSE
    std_train_mae = np.std(train_mae_list) if len(train_mae_list) > 1 else 0.0    # ✅ 新增MAE/RMSE
    mean_train_rmse = np.mean(train_rmse_list)  # ✅ 新增MAE/RMSE
    std_train_rmse = np.std(train_rmse_list) if len(train_rmse_list) > 1 else 0.0 # ✅ 新增MAE/RMSE
    mean_test_r2 = np.mean(test_r2_list)
    std_test_r2 = np.std(test_r2_list) if len(test_r2_list) > 1 else 0.0
    mean_gap = np.mean(gap_list)
    std_gap = np.std(gap_list) if len(gap_list) > 1 else 0.0
    mean_test_mae = np.mean(test_mae_list)
    std_test_mae = np.std(test_mae_list) if len(test_mae_list) > 1 else 0.0
    mean_test_rmse = np.mean(test_rmse_list)    # ✅ 新增MAE/RMSE
    std_test_rmse = np.std(test_rmse_list) if len(test_rmse_list) > 1 else 0.0   # ✅ 新增MAE/RMSE

    is_qualified = (mean_test_r2 >= CONFIG['min_r2_threshold'] and
                   mean_test_r2 <= CONFIG['max_r2_threshold'] and
                   mean_gap <= CONFIG['max_generalization_gap'])

    result = {
        'model_key': model_key,
        'group': 'Nonlinear',
        'model': model_name,
        'k': k,
        'features': feature_names,
        'mean_cv_r2': mean_cv_r2,
        'std_cv_r2': std_cv_r2,
        'mean_train_r2': mean_train_r2,
        'std_train_r2': std_train_r2,
        'mean_train_mae': mean_train_mae,      # ✅ 新增MAE/RMSE
        'std_train_mae': std_train_mae,        # ✅ 新增MAE/RMSE
        'mean_train_rmse': mean_train_rmse,    # ✅ 新增MAE/RMSE
        'std_train_rmse': std_train_rmse,      # ✅ 新增MAE/RMSE
        'mean_test_r2': mean_test_r2,
        'std_test_r2': std_test_r2,
        'mean_gap': mean_gap,
        'std_gap': std_gap,
        'mean_test_mae': mean_test_mae,
        'std_test_mae': std_test_mae,
        'mean_test_rmse': mean_test_rmse,      # ✅ 新增MAE/RMSE
        'std_test_rmse': std_test_rmse,        # ✅ 新增MAE/RMSE
        'n_evaluations': len(cv_r2_list),
        'convergence_warnings': convergence_warnings,
        'is_qualified': is_qualified,
        'is_deterministic': is_deterministic,
        'randomization_strategy': 'none'
    }

    if is_qualified:
        pred_file = save_detailed_predictions(
            model_key, y_train, first_pred_train,
            y_test, first_pred_test,
            train_indices, test_indices, timestamp
        )
        result['model_instance'] = first_model
        result['scaler'] = first_scaler
        result['predictions_file'] = pred_file

    return result


def stage6_nonlinear_group_final_training(X_train_stage2, X_test_stage2, y_train, y_test,
                                         train_indices, test_indices, nonlinear_combinations,
                                         nonlinear_group_indices):
    """阶段6：非线性组最终训练与评估（修复版 v3.1）"""
    print("\n" + "="*80)
    print("【阶段6】非线性组最终训练与评估（修复版 v3.1）")
    print("="*80)

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    cv_config = CONFIG['cv_strategy']['stage6']
    print(f"✅ CV策略: {cv_config['method']} ({cv_config['n_splits']}折×{cv_config['n_repeats']}次)")
    print(f"🚀 并行线程数: {CONFIG['stage6_n_jobs']}个")
    print(f"✅ 随机化策略: None（移除样本扰动）")
    print(f"✅ 模型评估策略:")
    print(f"   - 确定性模型 ({', '.join(CONFIG['deterministic_models'])}): 只评估1次")
    print(f"   - 随机性模型 ({', '.join(CONFIG['stochastic_models'])}): 评估{CONFIG['n_repeat_seeds']}次")

    model_library = {
        'SVR': SVR(
            C=1.0, epsilon=0.1, kernel='rbf', gamma='scale',
            max_iter=CONFIG['convergence_check'].get('svr_max_iter', -1),
            tol=CONFIG['convergence_check'].get('svr_tol', 1e-3)
        ),
        'KNN': KNeighborsRegressor(
            n_neighbors=5, weights='distance', metric='minkowski', p=2
        ),
        'MLP': MLPRegressor(
            hidden_layer_sizes=(50,), activation='relu', solver='adam',
            alpha=0.01,
            max_iter=CONFIG['convergence_check']['mlp_max_iter'],
            early_stopping=CONFIG['convergence_check']['mlp_early_stopping'],
            validation_fraction=CONFIG['convergence_check']['mlp_validation_fraction'],
            tol=CONFIG['convergence_check']['mlp_tol'],
            n_iter_no_change=CONFIG['convergence_check']['mlp_n_iter_no_change'],
            random_state=CONFIG['random_state']
        ),
        'AdaBoost': AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=2),
            n_estimators=50, learning_rate=0.5, loss='linear',
            random_state=CONFIG['random_state']
        )
    }

    print("\n构建评估任务列表...")
    all_tasks = []

    for comb_idx, comb in enumerate(nonlinear_combinations):
        k = comb['k']
        feature_names = comb['features']
        feature_indices_in_subset = comb['feature_indices']

        feature_indices_in_stage2 = [nonlinear_group_indices[i] for i in feature_indices_in_subset]

        X_train_comb = X_train_stage2[:, feature_indices_in_stage2]
        X_test_comb = X_test_stage2[:, feature_indices_in_stage2]

        for model_name, base_model in model_library.items():
            task_args = (
                X_train_comb, X_test_comb, y_train, y_test,
                train_indices, test_indices, model_name, base_model,
                k, comb_idx, feature_names,
                cv_config, timestamp
            )
            all_tasks.append(task_args)

    total_tasks = len(all_tasks)
    print(f"✅ 共构建 {total_tasks} 个评估任务")
    print(f"   = {len(nonlinear_combinations)}个特征组合 × {len(model_library)}个模型")
    print(f"🚀 开始{CONFIG['stage6_n_jobs']}线程并行评估...\n")

    start_time = time.time()

    results_raw = Parallel(n_jobs=CONFIG['stage6_n_jobs'], backend='loky', verbose=10)(
        delayed(evaluate_single_nonlinear_model_stage6)(task) for task in all_tasks
    )

    elapsed_time = time.time() - start_time

    all_results = []
    all_qualified_models = []

    for result in results_raw:
        if result is None:
            continue

        convergence_warnings_count = len(result.get('convergence_warnings', []))
        deterministic_tag = "(确定性)" if result['is_deterministic'] else "(随机性)"

        # ✅ 新增MAE/RMSE：all_results 中加入完整训练集和测试集MAE/RMSE字段
        all_results.append({
            'model_key': result['model_key'],
            'group': result['group'],
            'model': result['model'],
            'k': result['k'],
            'mean_cv_r2': result['mean_cv_r2'],
            'std_cv_r2': result['std_cv_r2'],
            'mean_train_r2': result['mean_train_r2'],
            'std_train_r2': result['std_train_r2'],
            'mean_train_mae': result['mean_train_mae'],      # ✅ 新增MAE/RMSE
            'std_train_mae': result['std_train_mae'],        # ✅ 新增MAE/RMSE
            'mean_train_rmse': result['mean_train_rmse'],    # ✅ 新增MAE/RMSE
            'std_train_rmse': result['std_train_rmse'],      # ✅ 新增MAE/RMSE
            'mean_test_r2': result['mean_test_r2'],
            'std_test_r2': result['std_test_r2'],
            'mean_gap': result['mean_gap'],
            'std_gap': result['std_gap'],
            'mean_test_mae': result['mean_test_mae'],
            'std_test_mae': result['std_test_mae'],
            'mean_test_rmse': result['mean_test_rmse'],      # ✅ 新增MAE/RMSE
            'std_test_rmse': result['std_test_rmse'],        # ✅ 新增MAE/RMSE
            'n_evaluations': result['n_evaluations'],
            'convergence_warnings': convergence_warnings_count,
            'is_deterministic': result['is_deterministic'],
            'deterministic_tag': deterministic_tag,
            'randomization_strategy': result['randomization_strategy']
        })

        if result['is_qualified']:
            result['convergence_warnings_count'] = convergence_warnings_count
            all_qualified_models.append(result)

    print(f"\n" + "="*80)
    print("🚀 阶段6并行评估完成统计（非线性组 - 修复版）:")
    print("="*80)
    print(f"总评估任务数: {total_tasks}")
    print(f"有效评估数: {len(all_results)}")
    print(f"合格模型数: {len(all_qualified_models)}")
    print(f"⏱️  总耗时: {elapsed_time/60:.1f}分钟")
    print(f"⚡ 平均每任务: {elapsed_time/total_tasks:.1f}秒")

    model_counter = Counter([r['model'] for r in all_qualified_models])
    print(f"\n📊 按模型类型统计合格模型:")
    for model_name in model_library.keys():
        count = model_counter.get(model_name, 0)
        model_type = "(确定性)" if model_name in CONFIG['deterministic_models'] else "(随机性)"
        print(f"  {model_name:15s} {model_type:10s}: {count:3d}个")

    total_convergence_warnings = sum([r.get('convergence_warnings', 0) for r in all_results])
    if total_convergence_warnings > 0:
        print(f"\n⚠️  收敛性警告: {total_convergence_warnings}次")
    else:
        print(f"\n✅ 无收敛性警告")

    print("="*80)

    return all_results, all_qualified_models, timestamp


# ==================== 阶段7：排序与Top30筛选 ====================
def sort_models_by_test_r2_with_stability(qualified_models):
    """按测试集R²均值排序（含稳定性惩罚）"""
    print("\n" + "="*80)
    print("【模型排序】按测试集R²均值排序（含稳定性惩罚）")
    print("="*80)

    penalty = CONFIG['stability_penalty']
    print(f"\n排序策略:")
    print(f"  排序得分 = Mean_Test_R² - {penalty} × Std_Test_R²")
    print(f"  注：确定性模型(SVR/KNN)的Std自然为0")

    for model in qualified_models:
        model['sort_score'] = (model['mean_test_r2'] -
                               penalty * model['std_test_r2'])

    qualified_models_sorted = sorted(qualified_models,
                                     key=lambda x: x['sort_score'],
                                     reverse=True)

    print(f"\n✅ 排序完成! Top30模型预览:")
    print(f"{'='*80}")

    for rank, model in enumerate(qualified_models_sorted[:30], 1):
        convergence_str = ""
        convergence_count = model.get('convergence_warnings_count', 0)
        if convergence_count > 0:
            convergence_str = f" ⚠️({convergence_count}次)"

        deterministic_tag = "(确定性)" if model.get('is_deterministic', False) else ""

        print(f"{rank:>2}. {model['model_key']:<50} | "
              f"排序={model['sort_score']:.4f} | "
              f"Test R²={model['mean_test_r2']:.4f}±{model['std_test_r2']:.4f}{deterministic_tag}{convergence_str}")

    return qualified_models_sorted


# ✅ 新增MAE/RMSE：evaluate_on_true_test_set 新增训练集MAE/RMSE收集
def evaluate_on_true_test_set(model_info, X_train_full, y_train_full,
                               X_test_true, y_test_true, features_60):
    """在真实独立测试集上评估单个非线性模型（修复版 - 区分确定性/随机性）"""
    model_key = model_info['model_key']
    model_features = model_info['features']

    feature_indices = [features_60.index(feat) for feat in model_features]
    X_train_model = X_train_full[:, feature_indices]
    X_test_model = X_test_true[:, feature_indices]

    need_scaling = True

    model_name = model_info['model']
    is_deterministic = model_name in CONFIG['deterministic_models']

    if is_deterministic:
        n_evaluations = 1
        repeat_seeds = [CONFIG['base_random_state']]
    else:
        n_evaluations = CONFIG['n_repeat_seeds']
        repeat_seeds = [CONFIG['base_random_state'] + i for i in range(n_evaluations)]

    train_r2_list = []
    train_mae_list = []    # ✅ 新增MAE/RMSE
    train_rmse_list = []   # ✅ 新增MAE/RMSE
    test_r2_list = []
    test_mae_list = []
    test_rmse_list = []

    if model_name == 'SVR':
        base_model = SVR(
            C=1.0, epsilon=0.1, kernel='rbf', gamma='scale',
            max_iter=CONFIG['convergence_check'].get('svr_max_iter', -1),
            tol=CONFIG['convergence_check'].get('svr_tol', 1e-3)
        )
    elif model_name == 'KNN':
        base_model = KNeighborsRegressor(
            n_neighbors=5, weights='distance', metric='minkowski', p=2
        )
    elif model_name == 'MLP':
        base_model = MLPRegressor(
            hidden_layer_sizes=(50,), activation='relu', solver='adam',
            alpha=0.01,
            max_iter=CONFIG['convergence_check']['mlp_max_iter'],
            early_stopping=CONFIG['convergence_check']['mlp_early_stopping'],
            validation_fraction=CONFIG['convergence_check']['mlp_validation_fraction'],
            tol=CONFIG['convergence_check']['mlp_tol'],
            n_iter_no_change=CONFIG['convergence_check']['mlp_n_iter_no_change'],
            random_state=CONFIG['random_state']
        )
    elif model_name == 'AdaBoost':
        base_model = AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=2),
            n_estimators=50, learning_rate=0.5, loss='linear',
            random_state=CONFIG['random_state']
        )
    else:
        base_model = model_info['model_instance']

    for seed in repeat_seeds:
        model = clone(base_model)

        if not is_deterministic:
            if hasattr(model, 'random_state'):
                try:
                    model.set_params(random_state=seed)
                except ValueError:
                    pass

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_model)
        X_test_scaled = scaler.transform(X_test_model)

        try:
            model.fit(X_train_scaled, y_train_full)

            pred_train = model.predict(X_train_scaled)
            pred_test = model.predict(X_test_scaled)

            train_metrics = calculate_comprehensive_metrics(y_train_full, pred_train)
            test_metrics = calculate_comprehensive_metrics(y_test_true, pred_test)

            train_r2_list.append(train_metrics['r2'])
            train_mae_list.append(train_metrics['mae'])    # ✅ 新增MAE/RMSE
            train_rmse_list.append(train_metrics['rmse'])  # ✅ 新增MAE/RMSE
            test_r2_list.append(test_metrics['r2'])
            test_mae_list.append(test_metrics['mae'])
            test_rmse_list.append(test_metrics['rmse'])

        except Exception as e:
            continue

    if len(train_r2_list) == 0:
        return None

    result = {
        'model_key': model_key,
        'mean_train_r2_true': np.mean(train_r2_list),
        'std_train_r2_true': np.std(train_r2_list) if len(train_r2_list) > 1 else 0.0,
        'mean_train_mae_true': np.mean(train_mae_list),    # ✅ 新增MAE/RMSE
        'std_train_mae_true': np.std(train_mae_list) if len(train_mae_list) > 1 else 0.0,  # ✅ 新增MAE/RMSE
        'mean_train_rmse_true': np.mean(train_rmse_list),  # ✅ 新增MAE/RMSE
        'std_train_rmse_true': np.std(train_rmse_list) if len(train_rmse_list) > 1 else 0.0, # ✅ 新增MAE/RMSE
        'mean_test_r2_true': np.mean(test_r2_list),
        'std_test_r2_true': np.std(test_r2_list) if len(test_r2_list) > 1 else 0.0,
        'mean_test_mae_true': np.mean(test_mae_list),
        'std_test_mae_true': np.std(test_mae_list) if len(test_mae_list) > 1 else 0.0,
        'mean_test_rmse_true': np.mean(test_rmse_list),
        'std_test_rmse_true': np.std(test_rmse_list) if len(test_rmse_list) > 1 else 0.0,
        'true_gap': np.mean(train_r2_list) - np.mean(test_r2_list),
        'n_seeds': len(train_r2_list),
        'is_deterministic': is_deterministic
    }

    return result


def stage7_evaluate_top_models_on_true_test(top_models, X_train_stage2, X_test_stage2,
                                            y_train, y_test, features_60, timestamp):
    """阶段7：在真实独立测试集上评估Top100模型，并选出Top30（修复版）"""
    print("\n" + "="*80)
    print("【阶段7】Top100模型在独立测试集上评估（修复版 v3.1）")
    print("="*80)

    top100_models = top_models[:min(100, len(top_models))]
    print(f"选择Top{len(top100_models)}模型进行真实测试集评估...")

    print(f"\n解封测试集:")
    print(f"  训练集: {X_train_stage2.shape[0]} 样本")
    print(f"  测试集: {X_test_stage2.shape[0]} 样本（真实独立）")

    start_time = time.time()

    true_test_results = Parallel(n_jobs=CONFIG['stage6_n_jobs'], backend='loky', verbose=10)(
        delayed(evaluate_on_true_test_set)(
            model_info, X_train_stage2, y_train, X_test_stage2, y_test, features_60
        ) for model_info in top100_models
    )

    elapsed_time = time.time() - start_time

    top100_with_true_test = []

    for i, model_info in enumerate(top100_models):
        true_result = true_test_results[i]

        if true_result is None:
            continue

        convergence_count = model_info.get('convergence_warnings_count', 0)
        is_deterministic = model_info.get('is_deterministic', False)

        # ✅ 新增MAE/RMSE：combined 中加入完整训练集和测试集MAE/RMSE字段
        combined = {
            'rank_training': i + 1,
            'model_key': model_info['model_key'],
            'group': model_info['group'],
            'model': model_info['model'],
            'k': model_info['k'],
            'features': model_info['features'],
            'is_deterministic': is_deterministic,
            'randomization_strategy': model_info['randomization_strategy'],

            # 训练集留出验证性能
            'mean_cv_r2_training': model_info['mean_cv_r2'],
            'std_cv_r2_training': model_info['std_cv_r2'],
            'mean_train_r2_training': model_info['mean_train_r2'],
            'std_train_r2_training': model_info['std_train_r2'],
            'mean_train_mae_training': model_info['mean_train_mae'],    # ✅ 新增MAE/RMSE
            'std_train_mae_training': model_info['std_train_mae'],      # ✅ 新增MAE/RMSE
            'mean_train_rmse_training': model_info['mean_train_rmse'],  # ✅ 新增MAE/RMSE
            'std_train_rmse_training': model_info['std_train_rmse'],    # ✅ 新增MAE/RMSE
            'mean_test_r2_training': model_info['mean_test_r2'],
            'std_test_r2_training': model_info['std_test_r2'],
            'mean_test_mae_training': model_info['mean_test_mae'],
            'std_test_mae_training': model_info['std_test_mae'],
            'mean_test_rmse_training': model_info['mean_test_rmse'],    # ✅ 新增MAE/RMSE
            'std_test_rmse_training': model_info['std_test_rmse'],      # ✅ 新增MAE/RMSE
            'mean_gap_training': model_info['mean_gap'],
            'convergence_warnings': convergence_count,

            # 真实测试集性能
            'mean_train_r2_true': true_result['mean_train_r2_true'],
            'std_train_r2_true': true_result['std_train_r2_true'],
            'mean_train_mae_true': true_result['mean_train_mae_true'],    # ✅ 新增MAE/RMSE
            'std_train_mae_true': true_result['std_train_mae_true'],      # ✅ 新增MAE/RMSE
            'mean_train_rmse_true': true_result['mean_train_rmse_true'],  # ✅ 新增MAE/RMSE
            'std_train_rmse_true': true_result['std_train_rmse_true'],    # ✅ 新增MAE/RMSE
            'mean_test_r2_true': true_result['mean_test_r2_true'],
            'std_test_r2_true': true_result['std_test_r2_true'],
            'mean_test_mae_true': true_result['mean_test_mae_true'],
            'std_test_mae_true': true_result['std_test_mae_true'],
            'mean_test_rmse_true': true_result['mean_test_rmse_true'],
            'std_test_rmse_true': true_result['std_test_rmse_true'],
            'true_gap': true_result['true_gap'],

            'model_instance': model_info['model_instance'],
            'scaler': model_info.get('scaler', None)
        }

        top100_with_true_test.append(combined)

    print(f"\n✅ Top{len(top100_with_true_test)}真实测试集评估完成")
    print(f"⏱️  耗时: {elapsed_time/60:.1f}分钟")

    # ✅ 新增MAE/RMSE：Top100 Excel 中加入完整训练集和测试集MAE/RMSE列
    top100_data = []
    for model in top100_with_true_test:
        convergence_str = ""
        if model.get('convergence_warnings', 0) > 0:
            convergence_str = f"⚠️({model['convergence_warnings']}次)"

        deterministic_str = "(确定性)" if model.get('is_deterministic', False) else "(随机性)"

        top100_data.append({
            '训练排名': model['rank_training'],
            '模型标识': model['model_key'],
            '组别': model['group'],
            '模型类型': model['model'],
            'K值': model['k'],
            '模型属性': deterministic_str,
            '收敛警告': convergence_str,

            # 训练集留出验证性能
            'Train_R²_Training': f"{model['mean_train_r2_training']:.4f}±{model['std_train_r2_training']:.4f}",
            'Train_MAE_Training': f"{model['mean_train_mae_training']:.4f}±{model['std_train_mae_training']:.4f}",    # ✅ 新增MAE/RMSE
            'Train_RMSE_Training': f"{model['mean_train_rmse_training']:.4f}±{model['std_train_rmse_training']:.4f}", # ✅ 新增MAE/RMSE
            'Test_R²_Training': f"{model['mean_test_r2_training']:.4f}±{model['std_test_r2_training']:.4f}",
            'Test_MAE_Training': f"{model['mean_test_mae_training']:.4f}±{model['std_test_mae_training']:.4f}",
            'Test_RMSE_Training': f"{model['mean_test_rmse_training']:.4f}±{model['std_test_rmse_training']:.4f}",    # ✅ 新增MAE/RMSE
            'Gap_Training': f"{model['mean_gap_training']:.4f}",
            'CV_R²_Training': f"{model['mean_cv_r2_training']:.4f}±{model['std_cv_r2_training']:.4f}",

            # 真实测试集性能
            'Train_R²_True': f"{model['mean_train_r2_true']:.4f}±{model['std_train_r2_true']:.4f}",
            'Train_MAE_True': f"{model['mean_train_mae_true']:.4f}±{model['std_train_mae_true']:.4f}",    # ✅ 新增MAE/RMSE
            'Train_RMSE_True': f"{model['mean_train_rmse_true']:.4f}±{model['std_train_rmse_true']:.4f}", # ✅ 新增MAE/RMSE
            'Test_R²_True': f"{model['mean_test_r2_true']:.4f}±{model['std_test_r2_true']:.4f}",
            'Test_MAE_True': f"{model['mean_test_mae_true']:.4f}±{model['std_test_mae_true']:.4f}",
            'Test_RMSE_True': f"{model['mean_test_rmse_true']:.4f}±{model['std_test_rmse_true']:.4f}",
            'Gap_True': f"{model['true_gap']:.4f}",

            '特征数量': len(model['features']),
            '特征列表': ', '.join(model['features'])
        })

    top100_df = pd.DataFrame(top100_data)
    top100_excel = os.path.join(top100_dir, f'top100_true_test_performance_{timestamp}.xlsx')

    with pd.ExcelWriter(top100_excel, engine='openpyxl') as writer:
        # Sheet1: 核心性能对比（含完整MAE/RMSE）
        top100_df[['训练排名', '模型标识', '模型类型', 'K值', '模型属性',
                   'Train_R²_Training', 'Train_MAE_Training', 'Train_RMSE_Training',
                   'Test_R²_Training', 'Test_MAE_Training', 'Test_RMSE_Training',
                   'Train_R²_True', 'Train_MAE_True', 'Train_RMSE_True',
                   'Test_R²_True', 'Test_MAE_True', 'Test_RMSE_True',
                   'Gap_Training', 'Gap_True']].to_excel(
            writer, sheet_name='核心性能对比', index=False
        )
        # Sheet2: 完整信息
        top100_df.to_excel(writer, sheet_name='完整信息', index=False)

    print(f"\n✅ Top{len(top100_with_true_test)}详细结果已保存: {top100_excel}")

    print(f"\n" + "="*80)
    print("从Top100中选择Top30（优化的筛选+排序策略）")
    print("="*80)

    qualified_for_top30 = []
    for model in top100_with_true_test:
        if (abs(model['true_gap']) < 0.25 and
            model['std_test_r2_true'] < 0.12):
            qualified_for_top30.append(model)

    if len(qualified_for_top30) < 30:
        print(f"\n⚠️  严格条件下只有{len(qualified_for_top30)}个模型合格，放宽筛选条件...")
        qualified_for_top30 = []
        for model in top100_with_true_test:
            if (abs(model['true_gap']) < 0.30 and
                model['std_test_r2_true'] < 0.15):
                qualified_for_top30.append(model)

    if len(qualified_for_top30) < 30:
        print(f"\n⚠️  放宽条件后仍只有{len(qualified_for_top30)}个，直接从Top100选择...")
        qualified_for_top30 = top100_with_true_test[:min(50, len(top100_with_true_test))]

    qualified_for_top30.sort(key=lambda x: x['mean_test_r2_true'], reverse=True)

    top30_models = qualified_for_top30[:min(30, len(qualified_for_top30))]

    for model in top30_models:
        composite_score = (0.7 * model['mean_test_r2_true'] -
                          0.2 * abs(model['true_gap']) -
                          0.1 * model['std_test_r2_true'])
        model['composite_score'] = composite_score

    print(f"\n✅ Top{len(top30_models)}模型已选出")
    print(f"\nTop{len(top30_models)}预览:")
    print(f"{'='*80}")

    for rank, model in enumerate(top30_models, 1):
        convergence_str = ""
        if model.get('convergence_warnings', 0) > 0:
            convergence_str = f" ⚠️({model['convergence_warnings']}次)"

        deterministic_str = "(确定性)" if model.get('is_deterministic', False) else ""

        print(f"{rank:>2}. {model['model_key']:<50} | "
              f"Test R²={model['mean_test_r2_true']:.4f}±{model['std_test_r2_true']:.4f} | "
              f"Gap={model['true_gap']:.4f} | "
              f"综合分={model['composite_score']:.4f}{deterministic_str}{convergence_str}")

    return top100_with_true_test, top30_models, timestamp


# ✅ 新增MAE/RMSE：save_top30_models_detailed 打印和保存中加入完整MAE/RMSE
def save_top30_models_detailed(top30_models, timestamp):
    """保存Top30模型详细信息（修复版）"""
    print("\n" + "="*80)
    print("【保存Top30模型详细信息】")
    print("="*80)

    top30_summary = []

    for rank, model_info in enumerate(top30_models, 1):
        model_key = model_info['model_key']

        print(f"\n{'='*80}")
        print(f"处理第{rank}名: {model_key}")
        print(f"{'='*80}")

        is_deterministic = model_info.get('is_deterministic', False)
        deterministic_str = "(确定性)" if is_deterministic else "(随机性)"
        print(f"  ✅ 模型属性: {deterministic_str}")

        convergence_warnings = model_info.get('convergence_warnings', 0)
        if convergence_warnings > 0:
            print(f"  ⚠️  收敛警告: {convergence_warnings}次")

        print(f"\n  📋 特征信息:")
        print(f"    特征数量: {len(model_info['features'])}个")
        print(f"    特征列表: {', '.join(model_info['features'])}")

        # ✅ 新增MAE/RMSE：打印训练集留出验证性能（含完整MAE/RMSE）
        print(f"\n  📊 训练集性能（留出验证）:")
        print(f"    Train R²   = {model_info['mean_train_r2_training']:.4f} ± {model_info['std_train_r2_training']:.4f}")
        print(f"    Train MAE  = {model_info['mean_train_mae_training']:.4f} ± {model_info['std_train_mae_training']:.4f}")    # ✅ 新增MAE/RMSE
        print(f"    Train RMSE = {model_info['mean_train_rmse_training']:.4f} ± {model_info['std_train_rmse_training']:.4f}") # ✅ 新增MAE/RMSE
        print(f"    CV R²      = {model_info['mean_cv_r2_training']:.4f} ± {model_info['std_cv_r2_training']:.4f}")
        print(f"    Test R²    = {model_info['mean_test_r2_training']:.4f} ± {model_info['std_test_r2_training']:.4f}")
        print(f"    Test MAE   = {model_info['mean_test_mae_training']:.4f} ± {model_info['std_test_mae_training']:.4f}")
        print(f"    Test RMSE  = {model_info['mean_test_rmse_training']:.4f} ± {model_info['std_test_rmse_training']:.4f}")   # ✅ 新增MAE/RMSE
        print(f"    Gap        = {model_info['mean_gap_training']:.4f}")

        # ✅ 新增MAE/RMSE：打印真实测试集性能（含完整MAE/RMSE）
        print(f"\n  📊 真实独立测试集性能:")
        print(f"    Train R²   = {model_info['mean_train_r2_true']:.4f} ± {model_info['std_train_r2_true']:.4f}")
        print(f"    Train MAE  = {model_info['mean_train_mae_true']:.4f} ± {model_info['std_train_mae_true']:.4f}")    # ✅ 新增MAE/RMSE
        print(f"    Train RMSE = {model_info['mean_train_rmse_true']:.4f} ± {model_info['std_train_rmse_true']:.4f}") # ✅ 新增MAE/RMSE
        print(f"    Test R²    = {model_info['mean_test_r2_true']:.4f} ± {model_info['std_test_r2_true']:.4f}")
        print(f"    Test MAE   = {model_info['mean_test_mae_true']:.4f} ± {model_info['std_test_mae_true']:.4f}")
        print(f"    Test RMSE  = {model_info['mean_test_rmse_true']:.4f} ± {model_info['std_test_rmse_true']:.4f}")
        print(f"    True Gap   = {model_info['true_gap']:.4f}")

        if is_deterministic:
            print(f"\n  💡 注：该模型为确定性算法，Std=0为正常现象")

        print(f"\n  📊 综合评估:")
        print(f"    综合得分 = {model_info['composite_score']:.4f}")

        # ✅ 新增MAE/RMSE：pkl文件中加入完整MAE/RMSE字段
        model_pkl_filename = f"rank{rank:02d}_{model_key}_model.pkl"
        model_pkl_filepath = os.path.join(top30_models_dir, model_pkl_filename)
        with open(model_pkl_filepath, 'wb') as f:
            pickle.dump({
                'model': model_info['model_instance'],
                'scaler': model_info.get('scaler', None),
                'features': model_info['features'],
                'model_name': model_info['model'],
                'group': model_info['group'],
                'k': model_info['k'],
                'is_deterministic': is_deterministic,

                # 训练集留出验证性能
                'mean_cv_r2_training': model_info['mean_cv_r2_training'],
                'std_cv_r2_training': model_info['std_cv_r2_training'],
                'mean_train_r2_training': model_info['mean_train_r2_training'],
                'std_train_r2_training': model_info['std_train_r2_training'],
                'mean_train_mae_training': model_info['mean_train_mae_training'],    # ✅ 新增MAE/RMSE
                'std_train_mae_training': model_info['std_train_mae_training'],      # ✅ 新增MAE/RMSE
                'mean_train_rmse_training': model_info['mean_train_rmse_training'],  # ✅ 新增MAE/RMSE
                'std_train_rmse_training': model_info['std_train_rmse_training'],    # ✅ 新增MAE/RMSE
                'mean_test_r2_training': model_info['mean_test_r2_training'],
                'std_test_r2_training': model_info['std_test_r2_training'],
                'mean_test_mae_training': model_info['mean_test_mae_training'],
                'std_test_mae_training': model_info['std_test_mae_training'],
                'mean_test_rmse_training': model_info['mean_test_rmse_training'],    # ✅ 新增MAE/RMSE
                'std_test_rmse_training': model_info['std_test_rmse_training'],      # ✅ 新增MAE/RMSE
                'mean_gap_training': model_info['mean_gap_training'],
                'convergence_warnings': convergence_warnings,

                # 真实测试集性能
                'mean_train_r2_true': model_info['mean_train_r2_true'],
                'std_train_r2_true': model_info['std_train_r2_true'],
                'mean_train_mae_true': model_info['mean_train_mae_true'],    # ✅ 新增MAE/RMSE
                'std_train_mae_true': model_info['std_train_mae_true'],      # ✅ 新增MAE/RMSE
                'mean_train_rmse_true': model_info['mean_train_rmse_true'],  # ✅ 新增MAE/RMSE
                'std_train_rmse_true': model_info['std_train_rmse_true'],    # ✅ 新增MAE/RMSE
                'mean_test_r2_true': model_info['mean_test_r2_true'],
                'std_test_r2_true': model_info['std_test_r2_true'],
                'mean_test_mae_true': model_info['mean_test_mae_true'],
                'std_test_mae_true': model_info['std_test_mae_true'],
                'mean_test_rmse_true': model_info['mean_test_rmse_true'],
                'std_test_rmse_true': model_info['std_test_rmse_true'],
                'true_gap': model_info['true_gap'],

                'composite_score': model_info['composite_score'],
                'randomization_strategy': 'none'
            }, f)

        print(f"\n  💾 文件保存:")
        print(f"    模型文件: {model_pkl_filename}")

        convergence_str = ""
        if convergence_warnings > 0:
            convergence_str = f"⚠️({convergence_warnings}次)"

        # ✅ 新增MAE/RMSE：top30_summary中加入完整MAE/RMSE字段
        top30_summary.append({
            '最终排名': rank,
            '训练排名': model_info.get('rank_training', '-'),
            '模型标识': model_key,
            '组别': model_info['group'],
            'K值': model_info['k'],
            '模型类型': model_info['model'],
            '模型属性': deterministic_str,
            '收敛警告': convergence_str,
            '综合得分': f"{model_info['composite_score']:.6f}",

            # 训练集留出验证性能
            'Train_R²_Training': f"{model_info['mean_train_r2_training']:.6f}",
            'Std_Train_R²_Training': f"{model_info['std_train_r2_training']:.6f}",
            'Train_MAE_Training': f"{model_info['mean_train_mae_training']:.6f}",    # ✅ 新增MAE/RMSE
            'Std_Train_MAE_Training': f"{model_info['std_train_mae_training']:.6f}", # ✅ 新增MAE/RMSE
            'Train_RMSE_Training': f"{model_info['mean_train_rmse_training']:.6f}",  # ✅ 新增MAE/RMSE
            'Std_Train_RMSE_Training': f"{model_info['std_train_rmse_training']:.6f}",# ✅ 新增MAE/RMSE
            'CV_R²_Training': f"{model_info['mean_cv_r2_training']:.6f}",
            'Std_CV_R²_Training': f"{model_info['std_cv_r2_training']:.6f}",
            'Test_R²_Training': f"{model_info['mean_test_r2_training']:.6f}",
            'Std_Test_R²_Training': f"{model_info['std_test_r2_training']:.6f}",
            'Test_MAE_Training': f"{model_info['mean_test_mae_training']:.6f}",
            'Std_Test_MAE_Training': f"{model_info['std_test_mae_training']:.6f}",
            'Test_RMSE_Training': f"{model_info['mean_test_rmse_training']:.6f}",    # ✅ 新增MAE/RMSE
            'Std_Test_RMSE_Training': f"{model_info['std_test_rmse_training']:.6f}", # ✅ 新增MAE/RMSE
            'Gap_Training': f"{model_info['mean_gap_training']:.6f}",

            # 真实测试集性能
            'Train_R²_True': f"{model_info['mean_train_r2_true']:.6f}",
            'Std_Train_R²_True': f"{model_info['std_train_r2_true']:.6f}",
            'Train_MAE_True': f"{model_info['mean_train_mae_true']:.6f}",    # ✅ 新增MAE/RMSE
            'Std_Train_MAE_True': f"{model_info['std_train_mae_true']:.6f}", # ✅ 新增MAE/RMSE
            'Train_RMSE_True': f"{model_info['mean_train_rmse_true']:.6f}",  # ✅ 新增MAE/RMSE
            'Std_Train_RMSE_True': f"{model_info['std_train_rmse_true']:.6f}",# ✅ 新增MAE/RMSE
            'Test_R²_True': f"{model_info['mean_test_r2_true']:.6f}",
            'Std_Test_R²_True': f"{model_info['std_test_r2_true']:.6f}",
            'Test_MAE_True': f"{model_info['mean_test_mae_true']:.6f}",
            'Std_Test_MAE_True': f"{model_info['std_test_mae_true']:.6f}",
            'Test_RMSE_True': f"{model_info['mean_test_rmse_true']:.6f}",
            'Std_Test_RMSE_True': f"{model_info['std_test_rmse_true']:.6f}",
            'Gap_True': f"{model_info['true_gap']:.6f}",

            '特征数量': len(model_info['features']),
            '特征列表': ', '.join(model_info['features'])
        })

    # ✅ 新增MAE/RMSE：Top30 Excel 中加入完整MAE/RMSE列
    top30_df = pd.DataFrame(top30_summary)
    excel_path = os.path.join(top30_dir, f'top30_models_summary_{timestamp}.xlsx')

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet1: 核心性能对比（含完整MAE/RMSE）
        top30_df[['最终排名', '训练排名', '模型标识', '组别', 'K值', '模型类型', '模型属性', '收敛警告',
                 '综合得分',
                 'Train_R²_Training', 'Train_MAE_Training', 'Train_RMSE_Training',
                 'CV_R²_Training', 'Test_R²_Training', 'Test_MAE_Training', 'Test_RMSE_Training',
                 'Gap_Training',
                 'Train_R²_True', 'Train_MAE_True', 'Train_RMSE_True',
                 'Test_R²_True', 'Test_MAE_True', 'Test_RMSE_True',
                 'Gap_True']].to_excel(
            writer, sheet_name='核心性能对比', index=False
        )
        # Sheet2: 特征信息
        top30_df[['最终排名', '模型标识', '特征数量', '特征列表']].to_excel(
            writer, sheet_name='特征信息', index=False
        )
        # Sheet3: 完整信息
        top30_df.to_excel(writer, sheet_name='完整信息', index=False)

    print(f"\n✅ Top{len(top30_models)}汇总Excel已保存: {excel_path}")

    return excel_path


# ==================== 阶段8：排序一致性验证 ====================
def evaluate_model_on_full_data_cv(model_info, X_full, y_full, features_60):
    """在全部数据上进行交叉验证评估单个非线性模型（修复版）"""
    model_key = model_info['model_key']
    model_features = model_info['features']

    feature_indices = [features_60.index(feat) for feat in model_features]
    X_model = X_full[:, feature_indices]

    need_scaling = True
    cv_config = CONFIG['cv_strategy']['stage8']

    model_name = model_info['model']
    is_deterministic = model_name in CONFIG['deterministic_models']

    if model_name == 'SVR':
        base_model = SVR(
            C=1.0, epsilon=0.1, kernel='rbf', gamma='scale',
            max_iter=CONFIG['convergence_check'].get('svr_max_iter', -1),
            tol=CONFIG['convergence_check'].get('svr_tol', 1e-3)
        )
    elif model_name == 'KNN':
        base_model = KNeighborsRegressor(
            n_neighbors=5, weights='distance', metric='minkowski', p=2
        )
    elif model_name == 'MLP':
        base_model = MLPRegressor(
            hidden_layer_sizes=(50,), activation='relu', solver='adam',
            alpha=0.01,
            max_iter=CONFIG['convergence_check']['mlp_max_iter'],
            early_stopping=CONFIG['convergence_check']['mlp_early_stopping'],
            validation_fraction=CONFIG['convergence_check']['mlp_validation_fraction'],
            tol=CONFIG['convergence_check']['mlp_tol'],
            n_iter_no_change=CONFIG['convergence_check']['mlp_n_iter_no_change'],
            random_state=CONFIG['random_state']
        )
    elif model_name == 'AdaBoost':
        base_model = AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=2),
            n_estimators=50, learning_rate=0.5, loss='linear',
            random_state=CONFIG['random_state']
        )
    else:
        base_model = model_info['model_instance']

    try:
        cv_splitter = RepeatedKFold(
            n_splits=cv_config['n_splits'],
            n_repeats=cv_config['n_repeats'],
            random_state=cv_config['random_state']
        )

        cv_scores = []

        for fold_idx, (train_idx, val_idx) in enumerate(cv_splitter.split(X_model)):
            X_train_fold = X_model[train_idx]
            X_val_fold = X_model[val_idx]
            y_train_fold = y_full[train_idx]
            y_val_fold = y_full[val_idx]

            model = clone(base_model)
            seed = CONFIG['random_state'] + fold_idx

            if not is_deterministic:
                if hasattr(model, 'random_state'):
                    try:
                        model.set_params(random_state=seed)
                    except ValueError:
                        pass

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_fold)
            X_val_scaled = scaler.transform(X_val_fold)

            try:
                model.fit(X_train_scaled, y_train_fold)
                val_pred = model.predict(X_val_scaled)
                val_score = r2_score(y_val_fold, val_pred)
                cv_scores.append(val_score)
            except Exception as e:
                continue

        if len(cv_scores) > 0:
            return np.mean(cv_scores)
        else:
            return None

    except Exception as e:
        print(f"  ⚠️  {model_key} 全数据CV失败: {str(e)}")
        return None


def stage8_ranking_consistency_verification(top30_models, X_full, y_full, features_60, timestamp):
    """阶段8：排序一致性验证（修复版 v3.1）"""
    print("\n" + "="*80)
    print("【阶段8】排序一致性验证 - 核心创新（修复版 v3.1）")
    print("="*80)

    print(f"\n目的: 验证Top{len(top30_models)}在全数据上的排序稳定性")
    print(f"输入: Top{len(top30_models)}模型架构 + 全部{X_full.shape[0]}样本")
    print(f"方法: RepeatedKFold交叉验证（{CONFIG['cv_strategy']['stage8']['n_splits']}折×{CONFIG['cv_strategy']['stage8']['n_repeats']}次）")

    print(f"\n开始在全数据（{X_full.shape[0]}样本）上交叉验证Top{len(top30_models)}模型...")

    start_time = time.time()

    cv_full_results = Parallel(n_jobs=CONFIG['stage6_n_jobs'], backend='loky', verbose=10)(
        delayed(evaluate_model_on_full_data_cv)(
            model_info, X_full, y_full, features_60
        ) for model_info in top30_models
    )

    elapsed_time = time.time() - start_time

    print(f"\n✅ 全数据交叉验证完成")
    print(f"⏱️  耗时: {elapsed_time/60:.1f}分钟")

    ranking_comparison = []

    for i, model_info in enumerate(top30_models):
        cv_full_score = cv_full_results[i]

        if cv_full_score is None:
            cv_full_score = -np.inf

        convergence_warnings = model_info.get('convergence_warnings', 0)
        is_deterministic = model_info.get('is_deterministic', False)

        ranking_comparison.append({
            'model_key': model_info['model_key'],
            'group': model_info['group'],
            'model': model_info['model'],
            'k': model_info['k'],
            'features': model_info['features'],
            'rank_test_subset': i + 1,
            'test_r2_subset': model_info['mean_test_r2_true'],
            'cv_r2_full': cv_full_score,
            'convergence_warnings': convergence_warnings,
            'is_deterministic': is_deterministic,
            'randomization_strategy': 'none'
        })

    ranking_comparison_sorted = sorted(ranking_comparison,
                                      key=lambda x: x['cv_r2_full'],
                                      reverse=True)

    for new_rank, item in enumerate(ranking_comparison_sorted, 1):
        item['rank_cv_full'] = new_rank

    ranking_comparison_sorted_by_test = sorted(ranking_comparison,
                                              key=lambda x: x['rank_test_subset'])

    for item in ranking_comparison_sorted_by_test:
        model_key = item['model_key']
        for sorted_item in ranking_comparison_sorted:
            if sorted_item['model_key'] == model_key:
                item['rank_cv_full'] = sorted_item['rank_cv_full']
                break

        item['rank_change'] = abs(item['rank_test_subset'] - item['rank_cv_full'])
        item['rank_change_pct'] = (item['rank_change'] / len(top30_models)) * 100

    from scipy.stats import spearmanr

    ranks_test = [item['rank_test_subset'] for item in ranking_comparison_sorted_by_test]
    ranks_cv_full = [item['rank_cv_full'] for item in ranking_comparison_sorted_by_test]

    spearman_rho, spearman_p = spearmanr(ranks_test, ranks_cv_full)

    print(f"\n" + "="*80)
    print("📊 排序一致性分析结果（修复版 v3.1）")
    print("="*80)

    print(f"\n🔥 核心指标:")
    print(f"  Spearman相关系数 ρ = {spearman_rho:.4f} (p={spearman_p:.6f})")

    if spearman_rho > 0.85:
        consistency_rating = "A级 - 高度一致"
        consistency_conclusion = "✅ 排序稳定，确认Top30"
    elif spearman_rho > 0.70:
        consistency_rating = "B级 - 部分一致"
        consistency_conclusion = "⚠️  排序部分一致，需深度分析"
    else:
        consistency_rating = "C级 - 不稳定"
        consistency_conclusion = "❌ 排序不稳定，需重新评估"

    print(f"  排序一致性评级: {consistency_rating}")
    print(f"  结论: {consistency_conclusion}")

    avg_rank_change = np.mean([item['rank_change'] for item in ranking_comparison_sorted_by_test])
    print(f"\n  平均排名变化: Δ_Rank = {avg_rank_change:.2f}")

    highly_stable_count = sum(1 for item in ranking_comparison_sorted_by_test if item['rank_change'] <= 3)
    print(f"  高度稳定模型数: {highly_stable_count}/{len(top30_models)} (Δ_Rank≤3)")

    print(f"\n📋 Top{len(top30_models)}排名对比详表:")
    print(f"{'='*100}")
    print(f"{'测试集排名':>10} | {'全数据CV排名':>15} | {'排名变化':>10} | {'模型属性':>10} | {'收敛警告':>10} | {'模型标识':<40}")
    print(f"{'-'*100}")

    for item in ranking_comparison_sorted_by_test:
        change_symbol = ""
        if item['rank_change'] == 0:
            change_symbol = "="
        elif item['rank_cv_full'] < item['rank_test_subset']:
            change_symbol = "↑"
        else:
            change_symbol = "↓"

        convergence_str = ""
        if item['convergence_warnings'] > 0:
            convergence_str = f"⚠️({item['convergence_warnings']}次)"

        deterministic_str = "(确定性)" if item['is_deterministic'] else "(随机性)"

        print(f"{item['rank_test_subset']:>10} | {item['rank_cv_full']:>15} | "
              f"{change_symbol}{item['rank_change']:>9} | {deterministic_str:>10} | "
              f"{convergence_str:>10} | {item['model_key']:<40}")

    unstable_models = [item for item in ranking_comparison_sorted_by_test
                      if item['rank_change'] > 5]

    if len(unstable_models) > 0:
        print(f"\n⚠️  排名变化较大的模型 (Δ_Rank > 5):")
        print(f"{'='*100}")
        for item in unstable_models:
            convergence_str = ""
            if item['convergence_warnings'] > 0:
                convergence_str = f" ⚠️({item['convergence_warnings']}次)"

            deterministic_str = "(确定性)" if item['is_deterministic'] else "(随机性)"

            print(f"  {item['model_key']:<40} {deterministic_str:>10} | "
                  f"测试集排名={item['rank_test_subset']:>2} → "
                  f"全数据CV排名={item['rank_cv_full']:>2} | "
                  f"变化={item['rank_change']}{convergence_str}")
    else:
        print(f"\n✅ 所有模型排名变化 ≤ 5，稳定性良好")

    ranking_data = []

    for item in ranking_comparison_sorted_by_test:
        convergence_str = ""
        if item['convergence_warnings'] > 0:
            convergence_str = f"⚠️({item['convergence_warnings']}次)"

        deterministic_str = "(确定性)" if item['is_deterministic'] else "(随机性)"

        ranking_data.append({
            '测试集排名': item['rank_test_subset'],
            '全数据CV排名': item['rank_cv_full'],
            '排名变化量': item['rank_change'],
            '排名变化百分比': f"{item['rank_change_pct']:.1f}%",
            '模型标识': item['model_key'],
            '组别': item['group'],
            '模型类型': item['model'],
            'K值': item['k'],
            '模型属性': deterministic_str,
            '收敛警告': convergence_str,
            'Test_R²_测试集': f"{item['test_r2_subset']:.6f}",
            'CV_R²_全数据': f"{item['cv_r2_full']:.6f}",
            '特征数量': len(item['features']),
            '特征列表': ', '.join(item['features'])
        })

    ranking_df = pd.DataFrame(ranking_data)

    excel_path = os.path.join(stage8_dir, f'ranking_consistency_analysis_{timestamp}.xlsx')

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        ranking_df[['测试集排名', '全数据CV排名', '排名变化量', '排名变化百分比',
                   '模型标识', '组别', '模型类型', 'K值', '模型属性', '收敛警告',
                   'Test_R²_测试集', 'CV_R²_全数据']].to_excel(
            writer, sheet_name='排序对比总表', index=False
        )

        correlation_data = [{
            'Spearman_ρ': spearman_rho,
            'p值': spearman_p,
            '平均排名变化': avg_rank_change,
            '高度稳定模型数': highly_stable_count,
            '一致性评级': consistency_rating,
            '结论': consistency_conclusion
        }]
        correlation_df = pd.DataFrame(correlation_data)
        correlation_df.to_excel(writer, sheet_name='Spearman相关系数', index=False)

        if len(unstable_models) > 0:
            unstable_df = ranking_df[ranking_df['排名变化量'] > 5]
            unstable_df.to_excel(writer, sheet_name='排名不稳定模型', index=False)

        ranking_df.to_excel(writer, sheet_name='完整信息', index=False)

    print(f"\n✅ 排序一致性分析结果已保存: {excel_path}")

    return {
        'spearman_rho': spearman_rho,
        'spearman_p': spearman_p,
        'avg_rank_change': avg_rank_change,
        'highly_stable_count': highly_stable_count,
        'consistency_rating': consistency_rating,
        'consistency_conclusion': consistency_conclusion,
        'ranking_comparison': ranking_comparison_sorted_by_test,
        'excel_path': excel_path
    }


# ==================== 主程序 ====================
if __name__ == "__main__":
    print("="*80)
    print("程序4：非线性组完整流程（阶段0-8）- 修复版 v3.1 + 完整MAE/RMSE")
    print("="*80)
    print("\n专注模型：SVR / KNN / MLP / AdaBoost")
    print("\n✅ 核心优化：")
    print("  • 移除样本扰动（提升排序稳定性）")
    print("  • MLP: 单层50神经元 + L2正则(alpha=0.01)")
    print("  • AdaBoost: 50棵树 + 学习率0.5")
    print("  • 区分确定性/随机性模型")
    print("  • 新增训练集和测试集MAE/RMSE完整计算、打印和保存")
    print("="*80)

    # 阶段0：数据准备
    print("\n【阶段0】数据准备")
    df = pd.read_excel(CONFIG['data_file'])
    print(f"成功加载数据: {df.shape}")

    target_col = CONFIG['target_col']
    exclude_cols = CONFIG['exclude_cols']
    all_columns = df.columns.tolist()

    print("\n步骤0.1：识别原始数值特征")
    features_name = []
    for col in all_columns:
        if col != target_col and col not in exclude_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                features_name.append(col)

    print(f"  识别到 {len(features_name)} 个原始数值特征")

    print("\n步骤0.2：提取原始特征数据")
    X_all = df[features_name].values
    y_all = df[target_col].values.ravel()
    print(f"  原始数据形状: X{X_all.shape}, y{y_all.shape}")

    print("\n步骤0.3：✅ 检查并移除 NaN 和 Inf/-Inf")
    X_all, y_all, valid_indices = check_and_remove_invalid_values(
        X_all, y_all, "原始数据"
    )
    print(f"  清洗后数据: X{X_all.shape}, y{y_all.shape}")

    df_clean = df.iloc[valid_indices].reset_index(drop=True)

    print("\n步骤0.4：分层采样数据分割")
    stratify_labels = create_stratify_bins(y_all, n_bins=5)

    print("\n步骤0.5：划分训练/测试集")
    X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
        X_all, y_all, np.arange(len(y_all)),
        test_size=CONFIG['test_size'],
        stratify=stratify_labels,
        random_state=CONFIG['random_state']
    )

    print(f"  训练集: {X_train.shape[0]} 样本 × {X_train.shape[1]} 特征")
    print(f"  测试集: {X_test.shape[0]} 样本 × {X_test.shape[1]} 特征")

    print("\n步骤0.6：特征分级标记")
    feature_tier_map = assign_feature_tiers(features_name)

    print("\n" + "="*80)
    print("✅ 数据准备完成")
    print("="*80)
    print("=== 测试集封存，阶段1-6仅在训练集上操作 ===")
    print("="*80)

    # 阶段1-2：特征筛选
    stage1_keep_indices, stage1_features = stage1_quick_filter(
        X_train, y_train, features_name, feature_tier_map
    )
    X_train_stage1 = X_train[:, stage1_keep_indices]
    X_test_stage1 = X_test[:, stage1_keep_indices]
    stage1_feature_tier_map = {feat: feature_tier_map[feat] for feat in stage1_features}

    stage2_keep_indices, stage2_features = stage2_integrated_selection(
        X_train_stage1, y_train, stage1_features, stage1_feature_tier_map
    )
    X_train_stage2 = X_train_stage1[:, stage2_keep_indices]
    X_test_stage2 = X_test_stage1[:, stage2_keep_indices]
    stage2_feature_tier_map = {feat: stage1_feature_tier_map[feat] for feat in stage2_features}

    print("\n" + "="*80)
    print("✅ 特征筛选完成（阶段1-2）")
    print("="*80)
    print(f"最终进入模型训练: {len(stage2_features)}个特征")

    # 阶段4：非线性组特征选择
    nonlinear_group_data = stage4_nonlinear_model_selection(
        X_train_stage2, y_train, stage2_features, stage2_feature_tier_map
    )

    # 阶段5：穷举搜索
    nonlinear_group_indices = nonlinear_group_data['indices']
    nonlinear_group_features = nonlinear_group_data['features']
    X_train_nonlinear = X_train_stage2[:, nonlinear_group_indices]

    nonlinear_combinations = stage5_nonlinear_group_exhaustive_search(
        X_train_nonlinear, y_train, nonlinear_group_features
    )

    # 阶段6：最终训练
    all_results, all_qualified_models, timestamp = stage6_nonlinear_group_final_training(
        X_train_stage2, X_test_stage2, y_train, y_test,
        train_indices, test_indices, nonlinear_combinations, nonlinear_group_indices
    )

    # 阶段7：结果输出与排序
    print("\n" + "="*80)
    print("【阶段7】结果输出、Top100评估与Top30筛选")
    print("="*80)

    # ✅ 新增MAE/RMSE：all_results CSV 中包含完整MAE/RMSE列
    all_results_df = pd.DataFrame(all_results)
    csv_path = os.path.join(CONFIG['output_dir'], f'all_model_results_{timestamp}.csv')
    all_results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n所有模型结果CSV已保存: {csv_path}")

    if all_qualified_models:
        print(f"\n原始合格模型数: {len(all_qualified_models)}个")

        all_qualified_models_sorted = sort_models_by_test_r2_with_stability(all_qualified_models)

        top100_with_true_test, top30_models, timestamp = stage7_evaluate_top_models_on_true_test(
            all_qualified_models_sorted, X_train_stage2, X_test_stage2,
            y_train, y_test, stage2_features, timestamp
        )

        top30_excel_path = save_top30_models_detailed(top30_models, timestamp)

        model_counter = Counter([m['model'] for m in all_qualified_models])

        print(f"\n" + "="*80)
        print("📊 按模型类型统计合格模型:")
        print("="*80)
        for model_name in ['SVR', 'KNN', 'MLP', 'AdaBoost']:
            count = model_counter.get(model_name, 0)
            model_type = "(确定性)" if model_name in CONFIG['deterministic_models'] else "(随机性)"
            print(f"  {model_name:15s} {model_type:10s}: {count:3d}个")

        total_convergence_warnings = sum([m.get('convergence_warnings_count', 0) for m in all_qualified_models])
        if total_convergence_warnings > 0:
            print(f"\n⚠️  总收敛警告次数: {total_convergence_warnings}")
        else:
            print(f"\n✅ 无收敛性警告")

        # 阶段8：排序一致性验证
        print("\n" + "="*80)
        print("准备进入阶段8 - 排序一致性验证...")
        print("="*80)

        X_full_stage2 = np.vstack([X_train_stage2, X_test_stage2])
        y_full = np.concatenate([y_train, y_test])

        print(f"\n全数据信息:")
        print(f"  总样本数: {X_full_stage2.shape[0]}")
        print(f"  特征数: {X_full_stage2.shape[1]}")

        stage8_results = stage8_ranking_consistency_verification(
            top30_models, X_full_stage2, y_full, stage2_features, timestamp
        )

        print(f"\n" + "="*80)
        print("📁 输出文件汇总:")
        print("="*80)
        print(f"1. 所有模型结果CSV: {csv_path}")
        print(f"2. Top100真实测试性能Excel: {os.path.join(top100_dir, f'top100_true_test_performance_{timestamp}.xlsx')}")
        print(f"3. Top30模型汇总Excel: {top30_excel_path}")
        print(f"4. Top30模型对象: {top30_models_dir}")
        print(f"5. 🔥 排序一致性分析Excel: {stage8_results['excel_path']}")

        # 最终总结
        print(f"\n" + "="*80)
        print("✅ 程序4执行完成! (非线性组 - 修复版 v3.1 + 完整MAE/RMSE)")
        print("="*80)

        print(f"\n🎯 关键成果汇总:")
        print(f"  • 合格模型总数: {len(all_qualified_models)}个")
        print(f"  • Top{len(top100_with_true_test)}模型已在真实测试集评估")
        print(f"  • Top{len(top30_models)}模型已确认")
        print(f"  • 排序一致性验证完成:")
        print(f"     - Spearman ρ = {stage8_results['spearman_rho']:.4f}")
        print(f"     - 评级: {stage8_results['consistency_rating']}")
        print(f"     - 结论: {stage8_results['consistency_conclusion']}")
        print(f"     - 高度稳定模型: {stage8_results['highly_stable_count']}/{len(top30_models)}")
        print(f"     - 平均排名变化: {stage8_results['avg_rank_change']:.2f}")

        # ✅ 新增MAE/RMSE：最终打印 Top1 完整指标
        print(f"\n  • ✅ Top1模型完整性能指标:")
        print(f"     - 模型:          {top30_models[0]['model']}")
        print(f"     - Train R²:      {top30_models[0]['mean_train_r2_true']:.4f}")
        print(f"     - Train MAE:     {top30_models[0]['mean_train_mae_true']:.4f}")    # ✅ 新增MAE/RMSE
        print(f"     - Train RMSE:    {top30_models[0]['mean_train_rmse_true']:.4f}")   # ✅ 新增MAE/RMSE
        print(f"     - Test R²:       {top30_models[0]['mean_test_r2_true']:.4f}")
        print(f"     - Test MAE:      {top30_models[0]['mean_test_mae_true']:.4f}")
        print(f"     - Test RMSE:     {top30_models[0]['mean_test_rmse_true']:.4f}")
        print(f"     - Gap:           {top30_models[0]['true_gap']:.4f}")

        print(f"\n" + "="*80)
        print("📊 优化效果对比:")
        print("="*80)
        print(f"  实际改进:")
        print(f"    • Spearman ρ: 0.11 → {stage8_results['spearman_rho']:.4f} "
              f"{'✅ 达标' if stage8_results['spearman_rho'] > 0.45 else '⚠️ 待优化'}")
        print(f"    • 平均排名变化: 8.80 → {stage8_results['avg_rank_change']:.2f} "
              f"{'✅ 达标' if stage8_results['avg_rank_change'] < 5 else '⚠️ 待优化'}")
        print(f"    • 高度稳定模型: 12/30 → {stage8_results['highly_stable_count']}/30 "
              f"{'✅ 改善' if stage8_results['highly_stable_count'] > 12 else '⚠️ 待优化'}")

        print("="*80)

    else:
        print("\n⚠️  无满足条件的模型")

    print(f"\n" + "="*80)
    print("✅ 程序4完整执行完毕! (非线性组修复版 v3.1 + 完整MAE/RMSE)")
    print("="*80)

### ✅ 程序4_非线性组_完整流程_修复版_v3.1_完整MAE_RMSE.py 完成 ###

检测到 12 个CPU核心
程序4：非线性组完整流程（阶段0-8）- 修复版 v3.1 + 完整MAE/RMSE

专注模型：SVR / KNN / MLP / AdaBoost

✅ 核心优化：
  • 移除样本扰动（提升排序稳定性）
  • MLP: 单层50神经元 + L2正则(alpha=0.01)
  • AdaBoost: 50棵树 + 学习率0.5
  • 区分确定性/随机性模型
  • 新增训练集和测试集MAE/RMSE完整计算、打印和保存

【阶段0】数据准备
成功加载数据: (202, 209)

步骤0.1：识别原始数值特征
  识别到 208 个原始数值特征

步骤0.2：提取原始特征数据
  原始数据形状: X(202, 208), y(202,)

步骤0.3：✅ 检查并移除 NaN 和 Inf/-Inf

  检查原始数据中的无效值...
    ✅ 数据clean，无无效值
  清洗后数据: X(202, 208), y(202,)

步骤0.4：分层采样数据分割

  创建分层标签(5个区间):
    区间 1: KQ范围 [3.40, 7.55], 样本数=41, 均值=6.59
    区间 2: KQ范围 [7.60, 9.59], 样本数=40, 均值=8.50
    区间 3: KQ范围 [9.60, 11.27], 样本数=40, 均值=10.51
    区间 4: KQ范围 [11.38, 13.47], 样本数=40, 均值=12.35
    区间 5: KQ范围 [13.49, 19.80], 样本数=41, 均值=15.15

步骤0.5：划分训练/测试集
  训练集: 161 样本 × 208 特征
  测试集: 41 样本 × 208 特征

步骤0.6：特征分级标记

==================== 特征分级标记 ====================

特征分级统计:
  S级: 4个 - S级(核心特征)
  A级: 22个 - A级(重要特征)
  B级: 182个 - B级(标准特征)

✅ 数据准备完成
=== 测试集封存，阶段1-6仅在训练集上操作 ===

【阶段1】快速粗筛 - 删除明显垃圾特征（纯规则）
输入特征数: 208

步骤1.1：删除常数/近常数特征
  条件

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Batch computation too fast (0.07395434379577637s.) Setting batch_size=2.
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    0.1s
[Parallel(n_jobs=6)]: Batch computation too slow (4.180784463882446s.) Setting batch_size=1.
[Parallel(n_jobs=6)]: Done  16 tasks      | elapsed:   48.6s
[Parallel(n_jobs=6)]: Done  27 tasks      | elapsed:   58.6s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:  1.1min
[Parallel(n_jobs=6)]: Done  50 tasks      | elapsed:  1.8min
[Parallel(n_jobs=6)]: Done  61 tasks      | elapsed:  2.0min
[Parallel(n_jobs=6)]: Done  78 out of  80 | elapsed:  2.9min remaining:    4.4s
[Parallel(n_jobs=6)]: Done  80 out of  80 | elapsed:  3.4min finished
[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Ba


🚀 阶段6并行评估完成统计（非线性组 - 修复版）:
总评估任务数: 80
有效评估数: 80
合格模型数: 27
⏱️  总耗时: 3.4分钟
⚡ 平均每任务: 2.6秒

📊 按模型类型统计合格模型:
  SVR             (确定性)     :  20个
  KNN             (确定性)     :   0个
  MLP             (随机性)     :   7个
  AdaBoost        (随机性)     :   0个

⚠️  收敛性警告: 83次

【阶段7】结果输出、Top100评估与Top30筛选

所有模型结果CSV已保存: D:\博一下\第一章主动学习\非线性组_结果_优化版_v3.1-4.22\all_model_results_20260423_095101.csv

原始合格模型数: 27个

【模型排序】按测试集R²均值排序（含稳定性惩罚）

排序策略:
  排序得分 = Mean_Test_R² - 0.1 × Std_Test_R²
  注：确定性模型(SVR/KNN)的Std自然为0

✅ 排序完成! Top30模型预览:
 1. Nonlinear_SVR_K5_Comb6                             | 排序=0.6477 | Test R²=0.6477±0.0000(确定性)
 2. Nonlinear_SVR_K6_Comb13                            | 排序=0.6440 | Test R²=0.6440±0.0000(确定性)
 3. Nonlinear_SVR_K6_Comb11                            | 排序=0.6426 | Test R²=0.6426±0.0000(确定性)
 4. Nonlinear_SVR_K7_Comb17                            | 排序=0.6414 | Test R²=0.6414±0.0000(确定性)
 5. Nonlinear_SVR_K7_Comb16                            | 排序=0.6372 | Test R²=0.6372±0.0000(确定性)
 6. No

[Parallel(n_jobs=6)]: Done  19 out of  27 | elapsed:    3.3s remaining:    1.3s
[Parallel(n_jobs=6)]: Done  22 out of  27 | elapsed:    3.4s remaining:    0.7s
[Parallel(n_jobs=6)]: Done  25 out of  27 | elapsed:    3.6s remaining:    0.2s
[Parallel(n_jobs=6)]: Done  27 out of  27 | elapsed:    6.1s finished
[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.



✅ Top27真实测试集评估完成
⏱️  耗时: 0.1分钟

✅ Top27详细结果已保存: D:\博一下\第一章主动学习\非线性组_结果_优化版_v3.1-4.22\top100_models\top100_true_test_performance_20260423_095101.xlsx

从Top100中选择Top30（优化的筛选+排序策略）

⚠️  严格条件下只有27个模型合格，放宽筛选条件...

⚠️  放宽条件后仍只有27个，直接从Top100选择...

✅ Top27模型已选出

Top27预览:
 1. Nonlinear_SVR_K5_Comb6                             | Test R²=0.6477±0.0000 | Gap=-0.1613 | 综合分=0.4211(确定性)
 2. Nonlinear_SVR_K6_Comb13                            | Test R²=0.6440±0.0000 | Gap=-0.1680 | 综合分=0.4172(确定性)
 3. Nonlinear_SVR_K6_Comb11                            | Test R²=0.6426±0.0000 | Gap=-0.1655 | 综合分=0.4167(确定性)
 4. Nonlinear_SVR_K7_Comb17                            | Test R²=0.6414±0.0000 | Gap=-0.1716 | 综合分=0.4147(确定性)
 5. Nonlinear_SVR_K7_Comb16                            | Test R²=0.6372±0.0000 | Gap=-0.1768 | 综合分=0.4107(确定性)
 6. Nonlinear_SVR_K4_Comb3                             | Test R²=0.6353±0.0000 | Gap=-0.1661 | 综合分=0.4115(确定性)
 7. Nonlinear_SVR_K6_Comb15                            | Test R²=0.62

[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Batch computation too fast (0.08110761642456055s.) Setting batch_size=2.
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  16 out of  27 | elapsed:    0.2s remaining:    0.1s
[Parallel(n_jobs=6)]: Done  22 out of  27 | elapsed:   15.1s remaining:    3.4s
[Parallel(n_jobs=6)]: Done  25 out of  27 | elapsed:   15.6s remaining:    1.2s



✅ 全数据交叉验证完成
⏱️  耗时: 0.5分钟

📊 排序一致性分析结果（修复版 v3.1）

🔥 核心指标:
  Spearman相关系数 ρ = 0.8449 (p=0.000000)
  排序一致性评级: B级 - 部分一致
  结论: ⚠️  排序部分一致，需深度分析

  平均排名变化: Δ_Rank = 3.11
  高度稳定模型数: 20/27 (Δ_Rank≤3)

📋 Top27排名对比详表:
     测试集排名 |         全数据CV排名 |       排名变化 |       模型属性 |       收敛警告 | 模型标识                                    
----------------------------------------------------------------------------------------------------
         1 |               1 | =        0 |      (确定性) |            | Nonlinear_SVR_K5_Comb6                  
         2 |               4 | ↓        2 |      (确定性) |            | Nonlinear_SVR_K6_Comb13                 
         3 |               3 | =        0 |      (确定性) |            | Nonlinear_SVR_K6_Comb11                 
         4 |               7 | ↓        3 |      (确定性) |            | Nonlinear_SVR_K7_Comb17                 
         5 |              17 | ↓       12 |      (确定性) |            | Nonlinear_SVR_K7_Comb16                 
         6 |          

[Parallel(n_jobs=6)]: Done  27 out of  27 | elapsed:   28.2s finished
